# CASMI26 engine + our ranker (share 0.1)

Built on [CASMI26 Next Direct W088](https://www.kaggle.com/code/llccqq624/casmi26-next-direct-w088)
by **Jiachen Li**, itself built on public work by **prvsiyan**, **haideptry** and
**megayak**. All credit for the engine belongs to them.

Two changes of ours: (1) tie-averaged rank features (LB 0.341 -> 0.349);
(2) the final ranking mixes in our own pipeline's rank with weight 0.1 --
a separate library search, candidate pool, fingerprint model and learned ranker
that is right on different molecules than the engine.


# CASMI26 Next Direct W088

## tl;dr

This public, self-contained GPU experiment directly regenerates predictions
from the competition data. It extends the scored 2026-09-19 two-ranker sweep
into the remaining public-ranker-weight interval.

## Context & Methods

- Controlled variable: `w_pv=0.88`
- Public 31-feature ranker weight: 0.88
- Expanded 51-feature simulated-ranker weight: 0.12
- All retrieval, candidate, model, seed, canonicalization and output settings
  are unchanged from the scored W80 pipeline.
- No upstream Notebook output is used as a data source.


# 🧪 CASMI 2026: Mastering the Domain Gap, Diagnostic Post-Mortem & Two-Ranker SOTA Engine
### *The Definitive Guide to timsTOF Domain Discrepancies, Candidate Traps, and the 0.353+ Two-Ranker Architecture*

> **Author:** [haideptry](https://www.kaggle.com/haideptry) (Author of the #1 Most-Voted Baseline & 0.339 Milestone)  
> **Topic:** Post-Mortem Analysis (Why LB Crashes from 0.339 to 0.297/0.327), Visual Domain Gap Diagnostics & The Two-Ranker SOTA Engine  
> **Hardware:** Runs completely offline on Dual T4 GPUs (`NvidiaTeslaT4x2`) or fast multi-threaded CPU.

---

## 🧭 Executive Summary: The Journey & Key Questions Answered

Across the CASMI 2026 competition, many teams observe confounding phenomena on the Public Leaderboard:
1. *Why does a notebook with 99% identical code score 0.339 in one run, but crash to 0.297 or 0.327 in another?*
2. *Why do tautomer deduplication and naive candidate caps destroy Leaderboard scores instead of boosting them?*
3. *Why does the public deep fingerprint model suffer from a subtle 0.49 vs 0.80 data leak, and how does a Two-Ranker Engine resolve it?*

This notebook is built in two integrated halves:
* **Part I — The Diagnostic Post-Mortem & Interactive Domain Gap Dashboard:** Pinpointing the exact chemistry and mathematical mechanisms behind the 3 lethal traps.
* **Part II — The Production Two-Ranker Adduct-Shifted Engine:** Full end-to-end execution combining `prvsiyan`'s held-out FPNet ranker with `megayak`'s 7-library adduct-shifted ranker to output a high-scoring `submission.csv`.

---

## 🏆 Community Credits & Standing on the Shoulders of Giants
* **prvsiyan**: For the pioneer *Analog Propagation* pipeline, the 6,930-bit FPNet Transformer architecture, and calibrated 31-feature ranking formulation.
* **megayak**: For discovering the deep FPNet training leak, simulating 2,250 molecules across 7 libraries (`megayak/casmi26-simulated-ranker-rows`), and introducing the Two-Ranker framework with adduct-shifted library search.
* **dariushafshar**: For pioneering metric-exact InChIKey14 evaluation canonicalization principles.
* **Enveda Biosciences**: For hosting this milestone dataset bridging natural product chemistry and machine learning.


# 🔬 Part I: Deconstructing the 3 Lethal Traps in CASMI 2026

### ⚠️ Trap 1: The Candidate Cap Recall Bug (The 0.339 $	o$ 0.297 Drop)
* **What happened**: When optimizing inference speed, capping candidates (e.g. `CAND_CAP = 80`) based on coarse score:
  $$\text{coarse\_score} = \text{lib\_sim} \times 100 - |\Delta \text{mass}|$$
* **The Fatal Flaw**: For Class 1 (library hits, $\text{lib\_sim} > 0$), this works fine. But for **Class 2** (no library hit, $\text{lib\_sim} = 0$), the coarse score reduces purely to $-|\Delta \text{mass}|$.
* In high-resolution mass spectrometry within an isobaric $\pm 8.5$ ppm window, whether an isomer deviates by $0.001$ Da or $0.003$ Da is completely random. Capping at 80 silently **eliminated 24.8% of true Class-2 reachable structures** before the ranker ever inspected them!

---

### ⚠️ Trap 2: The Premature Tautomer Deduplication Trap (The 0.339 $	o$ 0.327 Drop)
* **What happened**: In V20, candidates were aggressively canonicalized using `TautomerEnumerator().Canonicalize()` and deduplicated on InChIKey14 early in the candidate list.
* **The Fatal Flaw**: While the competition metric evaluates canonical InChIKey14, canonicalization alters ~10% of COCONUT identifiers. Applying it aggressively across raw top candidates **dropped 1,309 valid V17 candidate positions (13.1% of top-25 slots)**, replacing true high-probability isomers with lower-ranked decoys.

---

### ⚠️ Trap 3: The timsTOF Domain Gap & Deep FPNet Training Leak
* The test set is **100% timsTOF**. But the public FPNet model weights were pre-trained on GNPS, MoNA, MSDIAL, MassBank, and Pluskal structures.
* On those libraries, FPNet single-channel MRR looks unrealistically high (**0.76 – 0.82**), while on held-out timsTOF molecules (`enveda-np-examples`), its true MRR is **0.49**.
* **The Two-Ranker Solution**: We run two distinct rankers on the exact same evidence:
  1. **Ranker 1 (0.65 weight)**: prvsiyan's held-out timsTOF ranker + FPNet models.
  2. **Ranker 2 (0.35 weight)**: An extended ranker fitted on 2,250 molecules from 7 libraries without FPNet, powered by Adduct-Shifted matching and same-polarity analogs.


In [ ]:
# ===================================================================================
#  Diagnostic Visual Dashboard: Visualizing the Traps & the Domain Gap
# ===================================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Candidate Cap Recall Cliff
caps = [20, 40, 60, 80, 100, 150, 200, 300, 500, 'Uncapped (V17)']
recall_class2 = [0.42, 0.58, 0.69, 0.752, 0.82, 0.91, 0.96, 0.985, 0.998, 1.00]
lb_impact = [0.245, 0.268, 0.284, 0.297, 0.312, 0.328, 0.334, 0.337, 0.339, 0.339]

ax1 = axes[0]
color = '#e74c3c'
ax1.set_xlabel('Candidate Cap Threshold')
ax1.set_ylabel('Class 2 True Molecule Recall', color=color, fontweight='bold')
l1 = ax1.plot(range(len(caps)), recall_class2, 'o-', color=color, lw=2.5, label='Class-2 Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(len(caps)))
ax1.set_xticklabels([str(c) for c in caps], rotation=45, ha='right', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.3)

ax1_t = ax1.twinx()
color2 = '#2980b9'
ax1_t.set_ylabel('Public Leaderboard Score', color=color2, fontweight='bold')
l2 = ax1_t.plot(range(len(caps)), lb_impact, 's--', color=color2, lw=2, label='Public LB')
ax1_t.tick_params(axis='y', labelcolor=color2)
ax1.set_title('Trap 1: The Candidate Cap Recall Cliff\n(80 Cap Drops 24.8% of True Class-2 Answers)', fontweight='bold', fontsize=11)

# 2. FPNet Leak vs Held-out timsTOF MRR
ax2 = axes[1]
datasets = ['GNPS\n(Train seen)', 'MoNA\n(Train seen)', 'MSDIAL\n(Train seen)', 'MassBank\n(Train seen)', 'enveda-np\n(Held-out timsTOF)']
mrr_scores = [0.78, 0.81, 0.82, 0.76, 0.49]
colors = ['#95a5a6', '#95a5a6', '#95a5a6', '#95a5a6', '#e67e22']
bars = ax2.bar(datasets, mrr_scores, color=colors, edgecolor='black', alpha=0.85, width=0.6)
ax2.set_ylabel('Single-Channel MRR', fontweight='bold')
ax2.set_ylim(0, 1.0)
ax2.set_title('Trap 3: Deep FPNet Library Leakage\n(0.76-0.82 on seen vs 0.49 on held-out timsTOF)', fontweight='bold', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.3)
for b in bars:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{b.get_height():.2f}', ha='center', fontweight='bold', fontsize=9)

# 3. Adduct Shift Principle
ax3 = axes[2]
x_frag = np.array([120, 160, 210, 280])
ax3.vlines(x_frag, 0, [0.4, 0.9, 0.6, 0.3], colors='#2ecc71', lw=2.5, label='[M+H]+ Library Reference')
delta_shift = 21.98  # Na - H
ax3.vlines(x_frag + delta_shift, 0, [0.38, 0.88, 0.59, 0.28], colors='#9b59b6', lw=2.5, linestyle='--', label='[M+Na]+ Query (Precursor Shifted)')
ax3.set_xlabel('m/z')
ax3.set_ylabel('Normalized Intensity', fontweight='bold')
ax3.set_title('Adduct-Shift Principle\n(Neutral Loss Invariance: $\\Delta = prec_{query} - prec_{lib}$)', fontweight='bold', fontsize=11)
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


# 🚀 Part II: Production Two-Ranker Engine Execution (0.353+ SOTA)

We now execute the full, standalone Two-Ranker Engine:
1. **Unpack & Install Offline Wheels**: RDKit 2026.3.3.
2. **Build Neutral-Mass Reference Library & COCONUT 2.0 Pool** (711k candidates).
3. **Multi-Channel Evidence Extraction**:
   - Direct & Adduct-Shifted Entropy Matching (`search_shift_rows`).
   - Same-Polarity Analog Search (`pos` / `neg` representatives).
   - Adduct-Aware MetFrag-lite Charge Carrier Modeling.
   - Dual FPNet Neural Transformer Logits ($f \cdot z$).
4. **Two-Ranker Ensembling**:
   - Fit 8 GBDT models on `rank_train.npz` (weight 0.65).
   - Fit 4 GBDT models on 7-library `sim_rank_rows_nofp.npz` (weight 0.35).
   - Within-molecule reciprocal probability blending.
5. **Exact Evaluation Metric Canonicalization & Output Validation**:
   - Output exact 400 rows $\times$ 25 valid SMILES.


In [ ]:
import os, sys, glob, subprocess, time
T_START = time.time()
whl = sorted(glob.glob('/kaggle/input/**/rdkit-2026.3.3*.whl', recursive=True)) or sorted(glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True))
if whl:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], check=False)
import rdkit; print('RDKit', rdkit.__version__)
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=len)
    if not hits: raise FileNotFoundError(name)
    return hits[0]
COMP = os.path.dirname(find('test.parquet'))
print(COMP, os.listdir(COMP))
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')

## 1 · Similarity kernels, adducts, ranking features, MetFrag-lite
Entropy similarity (Li et al. 2021), mass-shifted search, adduct table and prvsiyan's 31 ranking features, plus `search_shift_rows` (adduct-shifted library match) and `explain_score_adduct`.

In [ ]:
%%writefile pv.py
"""prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied
verbatim where possible so local simulations and the Kaggle notebook share one source of truth.
Additions are marked `# ADDED`.
"""
import math
import numpy as np
from numba import njit, prange


class CFG:
    PPM_WIN = 10.0
    PPM_FALLBACK = 30.0
    INT_FLOOR = 0.002
    MAX_PEAKS = 256
    MZ_TOL = 0.01
    INT_POWER = 1.0
    ENT_WEIGHT = True
    ANALOG_WIN = 200.0
    N_ANALOG = 80
    SIM_POWER = 3.0
    W1_PRIORS = (0.30, 0.60)
    SEEDS = (0, 1, 2, 3)
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)
    TOPN = 25


# ----------------------------------------------------------------------------------------------
# similarity kernels
# ----------------------------------------------------------------------------------------------
@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx; c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64); j = 0
    for i in range(n):
        if it[i] >= thr: idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort(); idx = k2; c = topk
    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S; s2 = 0.0
            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi


@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol: buf[b] = qp[i]; i += 1; b += 1
        elif d > tol: buf[b] = cp[j]; j += 1; b += 1
        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)


@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out


@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001: return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)): sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b


# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)
@njit(cache=True, fastmath=True, parallel=True)
def search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):
    out = np.zeros(hi - lo, np.float32)
    for k in prange(hi - lo):
        a = roff[lo + k]; b = roff[lo + k + 1]
        if b <= a: continue
        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])
    return out


# ADDED: library search where each reference is shifted by (query precursor - reference precursor):
# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)
@njit(cache=True, fastmath=True, parallel=True)
def search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out


# ADDED: clean a list of spectra into a flat store
@njit(cache=True)
def clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):
    n = len(rows)
    lens = np.zeros(n + 1, np.int64)
    tmp_m = []
    tmp_i = []
    for k in range(n):
        r = rows[k]
        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)
        tmp_m.append(cm); tmp_i.append(cp)
        lens[k + 1] = lens[k] + len(cm)
    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)
    for k in range(n):
        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]
    return lens, om, oi


# ----------------------------------------------------------------------------------------------
# adducts
# ----------------------------------------------------------------------------------------------
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990; PROTON = MASS['H'] - E; H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']; FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']
ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}


def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m] * z - d) / n
    return out


# ----------------------------------------------------------------------------------------------
# ranking features (31)
# ----------------------------------------------------------------------------------------------
N_ANALOG = 80
P_SIM = 3.0
N_FEAT = 31


def _rank_norm(x):
    # ryqn00 variant: ties AVERAGED. The original ordered tied candidates by
    # their position in the list, which differs between training and serving.
    from scipy.stats import rankdata
    x = np.asarray(x)
    return (rankdata(-x, method='average') - 1) / max(1, len(x) - 1)


def _z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)


def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)
        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)
        ap = (tan * (w ** P_SIM)[None, :]).max(1)
        a1 = (tan * w[None, :]).max(1)
        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])
        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)
    else:
        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)
        nrm = raw / np.sqrt(np.maximum(cs, 1.0))
        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw == raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc, np.float32)] * 6
    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc, np.float32)] * 6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc, np.float32)] * 4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),
        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),
        np.full(nc, np.log(max(nc, 1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)


# ----------------------------------------------------------------------------------------------
# MetFrag-lite
# ----------------------------------------------------------------------------------------------
AMU = {'C': 12.0, 'H': 1.00782503207, 'N': 14.0030740048, 'O': 15.9949146196, 'P': 30.97376163,
       'S': 31.97207100, 'F': 18.99840322, 'Cl': 34.96885268, 'Br': 78.9183371, 'I': 126.904473,
       'Na': 22.9897692809, 'K': 38.96370668, 'Si': 27.9769265325, 'B': 11.0093054, 'Se': 79.9165213}
H = AMU['H']


def mol_graph(smi):
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n


def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps = []
    for s in range(n):
        if seen[s]: continue
        stack = [s]; seen[s] = True; cur = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps


def fragment_masses(smi, max_breaks=2, max_bonds=34):
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i + 1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))


def frag_masses_safe(smi):
    try: return fragment_masses(smi)
    except Exception: return np.zeros(0)


# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)
CARRIERS = {
    "[M+H]+": (PROTON,), "[M-H2O+H]+": (PROTON,), "[M-2H2O+H]+": (PROTON,),
    "[M+NH4]+": (PROTON, NH4 - E), "[M+Na]+": (PROTON, MASS['Na'] - E), "[M+K]+": (PROTON, MASS['K'] - E),
    "[M-H]-": (-PROTON,), "[M-H2O-H]-": (-PROTON,), "[M+CH2O2-H]-": (-PROTON, FORMATE - PROTON),
    "[M+Cl]-": (-PROTON, MASS['Cl'] + E),
}


def explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    car = CARRIERS.get(adduct, (PROTON,) if "+" in adduct[-2:] else (-PROTON,))
    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)


def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)

## 2 · Spectrum → fingerprint model
prvsiyan's FPNet; weights from `prvsiyan/casmi26-fp-models-v2` (single-spectrum and merged-spectrum models, averaged views).

In [ ]:
%%writefile pv_fp.py
"""Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch."""
import math
import numpy as np
# ----------------------------------------------------------------------------------------------
# spectrum -> fingerprint model
# ----------------------------------------------------------------------------------------------
import torch, torch.nn as nn, torch.nn.functional as F

FP_MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",
               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",
               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",
               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]
ADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]


def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4


def prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = it >= floor * mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        order = np.argsort(-it)
        bucket = (mz // win).astype(np.int64)
        cnt = {}; sel = []
        for i in order:
            b = bucket[i]; c = cnt.get(b, 0)
            if c < per_win: cnt[b] = c + 1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks - len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it / it.max())
    return mz.astype(np.float32), v.astype(np.float32)


class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer('inv', (2 * math.pi) / wav)

    def forward(self, x):
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)


class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h = h
        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, pad):
        B, N, D = x.shape; y = self.n1(x)
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        m = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d)
        self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))

    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B, N = mz.shape
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:, 0]
        msk = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))


def load_fp_models(paths, dev):
    single, merged = [], []
    nbits = None
    for pth in paths:
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        (merged if 'merged' in str(pth).replace('\\', '/').split('/')[-1] else single).append(net)
        nbits = ck['nbits']
    return single, merged, nbits


def merge_peaks(specs):
    """specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum."""
    specs = [(a, b) for a, b in specs if len(a)]
    if not specs: return np.zeros(0), np.zeros(0)
    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])
    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j] - mz[j - 1] < 0.005:
            if it[j] >= it[j - 1]: keep[j - 1] = False
            else: keep[j] = False
    return mz[keep], it[keep]


@torch.no_grad()
def logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):
    """P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode."""
    keep = [i for i, (a, _) in enumerate(P) if len(a)]
    if not keep: return None
    P = [P[i] for i in keep]
    B = len(P); N = max(len(a) for a, _ in P)
    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
    for i, (a, b) in enumerate(P):
        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
    T = lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad),
            T(np.asarray(prec, np.float32)[keep]),
            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX['<unk>']) for a in np.asarray(adduct, object)[keep]])),
            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),
            T(np.asarray(ce, np.float32)[keep]),
            T(np.asarray(mode, np.float32)[keep]))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)


def molecule_logits(models, specs, prec, adduct, instr, ce, mode):
    """prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).
    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum."""
    single, merged, dev = models
    out = []
    if single:
        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]
        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)
        if za is not None: out.append(za)
    if merged:
        mz, it = merge_peaks(specs)
        pm = float(np.median(prec))
        P = [prep_peaks(mz, it, pm)]
        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])
        if zb is not None: out.append(zb)
    return np.mean(out, axis=0) if out else None

## 3 · Engine
Library (row groups straight to float32), candidate pool (COCONUT ∪ training structures, ±10 ppm), index (all / positive / negative representatives), per-molecule channels, feature blocks, ranker and submission. The feature code is byte-for-byte the code that produced the training rows; a local check asserts both give identical matrices.

In [ ]:
%%writefile casmi_engine.py
"""End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,
but reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).

Stages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission
"""
import os, sys, glob, time, pickle, math
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa
import pv
from pv import CFG

POOL = None


class RANK:
    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged
    SEEDS = (0, 1)
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)

T0 = time.time()

# --- tie-averaged rank columns in the training rows (ryqn00 variant) ------------
# (rank column, raw column it ranks). Base block = prvsiyan's 31 features; ranker
# B appends libx (+31), anapol (+39), fragx (+44).
RANK_COLS_BASE = [(1, 0), (6, 5), (16, 15), (19, 18), (22, 21)]
RANK_COLS_B = RANK_COLS_BASE + [(32, 31), (35, 34), (40, 39), (45, 44), (48, 47)]


def retie(X, G, pairs):
    """Recompute rank columns per candidate list (G), ties averaged, as serving now
    does. Where a list has no ties the stored and new values must agree -- that
    proves each pair really is (rank of, raw)."""
    X = np.array(X, np.float32, copy=True)
    df = pd.DataFrame({"G": G})
    n = df.groupby("G").G.transform("size").to_numpy()
    for rc, vc in pairs:
        df["v"] = X[:, vc]
        new = (df.groupby("G").v.rank(ascending=False, method="average").to_numpy() - 1) / np.maximum(n - 1, 1)
        untied = df.groupby("G").v.transform("nunique").to_numpy() == n
        assert np.allclose(X[untied, rc], new[untied], atol=1e-5), f"column {rc} is not the rank of {vc}"
        X[:, rc] = new
    return X
LOCAL = os.environ.get("CASMI_LOCAL") == "1"
ROOTS = [r"C:\Users\HW-LEE\Desktop\CASMI"] if LOCAL else ["/kaggle/input"]


def find(name):
    for root in ROOTS:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    raise FileNotFoundError(name)


def log(msg):
    print(f"[{time.time()-T0:6.0f}s] {msg}", flush=True)


# ---------------------------------------------------------------------------------------------
# library
# ---------------------------------------------------------------------------------------------
def load_library(path):
    """Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB."""
    f = pq.ParquetFile(path)
    n_rows = f.metadata.num_rows
    npk = 0
    for i in range(f.num_row_groups):
        c = f.read_row_group(i, columns=["ms2_mzs"]).column(0).combine_chunks()
        npk += len(c.values)
    off = np.zeros(n_rows + 1, np.int64)
    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)
    prec, add, pol, ik, smi = [], [], [], [], []
    r0 = p0 = 0
    for i in range(f.num_row_groups):
        t = f.read_row_group(i, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode",
                                         "ms2_mzs", "ms2_normalized_intensities"])
        mzc = t.column("ms2_mzs").combine_chunks(); itc = t.column("ms2_normalized_intensities").combine_chunks()
        o = mzc.offsets.to_numpy().astype(np.int64)
        n = len(o) - 1; k = int(o[-1] - o[0])
        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]
        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]
        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])
        prec.append(t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64))
        add += t.column("adduct").cast(pa.string()).to_pylist()
        pol.append(np.array(t.column("ionization_mode").cast(pa.string()).to_pylist(), dtype=object) == "positive")
        ik += t.column("inchikey14").cast(pa.string()).to_pylist()
        smi += t.column("normalized_smiles").cast(pa.string()).to_pylist()
        r0 += n; p0 += k
        del t, mzc, itc
    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)
    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)
    nm = pv.neutral_mass(prec, add)
    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol,
                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))


# ---------------------------------------------------------------------------------------------
# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)
# ---------------------------------------------------------------------------------------------
_g = {}


def _fp_init():
    from rdkit import RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog("rdApp.*")
    _g["bits"] = np.load(find("fp_bits.npy"))
    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)


def fp_and_mass(smi):
    from rdkit import Chem
    from rdkit.Chem import MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([_g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g["bits"]]
        return np.packbits(fp), float(ExactMolWt(m))
    except Exception:
        return None


def canon_key(smi):
    from rdkit import Chem, RDLogger
    from rdkit.Chem.MolStandardize import rdMolStandardize
    RDLogger.DisableLog("rdApp.*")
    if "te" not in _g: _g["te"] = rdMolStandardize.TautomerEnumerator()
    try:
        m = Chem.MolFromSmiles(smi)
        return Chem.MolToInchiKey(_g["te"].Canonicalize(m))[:14] if m is not None else None
    except Exception:
        return None


def build_pool(L, workers):
    from multiprocessing import Pool as MPool
    d = os.path.dirname(find("coco_fp.npy"))
    cm = pickle.load(open(os.path.join(d, "coco_meta.pkl"), "rb"))
    co_fp = np.load(os.path.join(d, "coco_fp.npy")); co_mass = np.load(os.path.join(d, "coco_mass.npy"))
    co_keys = np.asarray(cm["keys"], dtype=object); co_smi = np.asarray(cm["smiles"], dtype=object)
    tr = pd.DataFrame({"ik": L["ik"], "smi": L["smi"]}).dropna().drop_duplicates("ik")
    tr = tr[~tr.ik.isin(set(co_keys))]
    log(f"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}")
    with MPool(workers) as mp:
        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)
    ok = np.array([r is not None for r in res])
    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])
    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])
    good = np.isfinite(mass)
    o = np.argsort(np.where(good, mass, 1e18), kind="mergesort")[: good.sum()]
    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])
    P["k2i"] = pd.Series(np.arange(len(o)), index=P["keys"]); P["k2i"] = P["k2i"][~P["k2i"].index.duplicated()]
    log(f"pool {len(o):,} structures")
    return P


def pool_fps(idx):
    return np.unpackbits(np.asarray(POOL["fp"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]


def pool_window(t, ppm):
    a = np.searchsorted(POOL["mass"], t * (1 - ppm / 1e6), "left")
    b = np.searchsorted(POOL["mass"], t * (1 + ppm / 1e6), "right")
    return np.arange(a, b)


# ---------------------------------------------------------------------------------------------
# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)
# ---------------------------------------------------------------------------------------------
def _reps(L, rows):
    npk = np.diff(L["off"])[rows]
    df = pd.DataFrame({"row": rows, "p": L["row_p"][rows], "npk": npk})
    df = df.sort_values(["p", "npk", "row"], ascending=[True, False, True]).drop_duplicates("p")
    rep = df.row.values; rep_nm = L["nm"][rep]
    o = np.argsort(rep_nm, kind="mergesort"); rep = rep[o]
    roff, rmz, rit = pv.clean_store(rep, L["off"], L["mz"], L["it"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
    return dict(rep=rep, rep_p=L["row_p"][rep], rep_nm=L["nm"][rep], roff=roff, rmz=rmz, rit=rit)


def build_index(L):
    ok = np.isfinite(L["nm"]) & (L["row_p"] >= 0)
    rows = np.where(ok)[0]
    o = np.argsort(L["nm"][rows], kind="mergesort")
    I = dict(lib_rows=rows[o], lib_nm=L["nm"][rows[o]])
    I["all"] = _reps(L, rows)
    I["pos"] = _reps(L, rows[L["pol"][rows] == 1])
    I["neg"] = _reps(L, rows[L["pol"][rows] == -1])
    log(f"index: {len(rows):,} library rows; representatives {len(I['all']['rep']):,}")
    return I


# ---------------------------------------------------------------------------------------------
# per-molecule channels (identical to e02_channels.molecule_channels)
# ---------------------------------------------------------------------------------------------
def clean(mz, it):
    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)


def analog_search(R, Q, target):
    lo = np.searchsorted(R["rep_nm"], target - CFG.ANALOG_WIN, "left")
    hi = np.searchsorted(R["rep_nm"], target + CFG.ANALOG_WIN, "right")
    asim = np.zeros(hi - lo, np.float32)
    if hi > lo and Q:
        shift = (target - R["rep_nm"][lo:hi]).astype(np.float32)
        for qm, qp in Q:
            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R["roff"], R["rmz"], R["rit"], CFG.MZ_TOL, shift))
    return R["rep_p"][lo:hi], asim


def top_analogs(p, s):
    if len(p) == 0: return []
    df = pd.DataFrame({"p": p, "s": s}).groupby("p", sort=False).s.max().sort_values(ascending=False, kind="mergesort")
    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))


def molecule_channels(L, I, specs, precs, pols, target):
    Q = [clean(a, b) for a, b in specs]
    keep = [i for i, (a, _) in enumerate(Q) if len(a)]
    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]
    tol = target * CFG.PPM_WIN / 1e6
    lo = np.searchsorted(I["lib_nm"], target - tol, "left"); hi = np.searchsorted(I["lib_nm"], target + tol, "right")
    crow = I["lib_rows"][lo:hi]
    u, inv = np.unique(L["row_p"][crow], return_inverse=True)
    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)
    for j, (qm, qp) in enumerate(Q):
        if len(crow) == 0: break
        s = pv.search(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                      CFG.INT_POWER, CFG.ENT_WEIGHT)
        np.maximum.at(M[j], inv, s)
        sh = (precs[j] - L["prec"][crow]).astype(np.float32)
        s2 = pv.search_shift_rows(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)
        np.maximum.at(MS[j], inv, s2)
    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)
    z = np.zeros(0, np.float32)
    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)
    p, s = analog_search(I["all"], Q, target)
    ana = top_analogs(p, s)
    ps, ss = [], []
    for sign, name in ((1, "pos"), (-1, "neg")):
        QQ = [q for q, pl in zip(Q, pols) if pl == sign]
        if QQ:
            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)
    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []
    return lib, ana, anapol


def parse_ce(v):
    try:
        a = np.abs(np.atleast_1d(np.asarray(v, float)))
        return float(a.mean()) if len(a) else 25.0
    except Exception:
        return 25.0




# ---------------------------------------------------------------------------------------------
# ranking features: prvsiyan's 31 + our blocks (identical to e05_rank.py BLOCKS)
# ---------------------------------------------------------------------------------------------
def _rel(v):
    v = np.asarray(v, np.float32)
    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]


def _analog_feats(cfp, ana):
    nc = len(cfp)
    if not ana: return np.zeros((nc, 5), np.float32)
    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)
    inter = cf @ afp.T
    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)
    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)
    ap = (tan * (w ** 3)[None, :]).max(1)
    ap6 = (tan * (w ** 6)[None, :]).max(1)
    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)


def blk_base(r, cfp):
    ana = r["ana"]
    afp = pool_fps([p for p, _ in ana]) if ana else None
    sims = np.array([s for _, s in ana], np.float32)
    return pv.rank_features(cfp, r["lib"], afp, sims, r["zlog"], r["frag"])


def blk_mass(r, cfp):
    ppm = (POOL["mass"][r["cand"]] - r["target"]) / r["target"] * 1e6
    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)


def blk_libx(r, cfp):
    cnt = np.log1p(r["lib_cnt"])
    return np.column_stack(_rel(r["lib_mean"]) + _rel(r["libsh"]) + [cnt, cnt - cnt.max()]).astype(np.float32)


def blk_anapol(r, cfp):
    return _analog_feats(cfp, r["anapol"])


def blk_fragx(r, cfp):
    return np.column_stack(_rel(r["frag_ad"]) + _rel(r["frag_mean"]) + [r["frag_pol"]]).astype(np.float32)


BLOCKS = {"base": blk_base, "mass": blk_mass, "libx": blk_libx, "anapol": blk_anapol, "fragx": blk_fragx}


# ---------------------------------------------------------------------------------------------
# run
# ---------------------------------------------------------------------------------------------
def compute_channels(L, I, models, te):
    mols = list(te.groupby("molecule_id", sort=True))
    recs = []
    for gi, (mid, sub) in enumerate(mols):
        specs = []
        for r in sub.itertuples():
            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)
            k_ = a_ <= float(r.precursor_mz) + 2.0
            a_, b_ = a_[k_], b_[k_]
            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))
        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)
        nms = nms[np.isfinite(nms)]
        rec = dict(mid=mid, cand=np.zeros(0, np.int64))
        if len(nms):
            target = float(np.median(nms))
            modes = np.where(sub.ionization_mode.astype(str).values == "positive", 1.0, -1.0)
            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)
            cand = pool_window(target, CFG.PPM_WIN)
            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)
            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)
            import pv_fp
            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),
                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),
                                         ces, modes)
            u, lmax, lmean, lsh, lcnt = lib
            pos = {int(x): i for i, x in enumerate(u)}
            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0
            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,
                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),
                       ana=ana, anapol=anapol)
            for nm_, arr in (("lib", lmax), ("lib_mean", lmean), ("libsh", lsh), ("lib_cnt", lcnt)):
                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v
        recs.append(rec)
        if gi % 50 == 0: log(f"  channels {gi}/{len(mols)}")
    return mols, recs


def compute_frag(recs, workers):
    from multiprocessing import Pool as MPool
    uc = np.unique(np.concatenate([r["cand"] for r in recs]))
    log(f"MetFrag-lite on {len(uc):,} candidates")
    with MPool(workers) as mp:
        fr = mp.map(pv.frag_masses_safe, list(POOL["smiles"][uc]), chunksize=16)
    fmap = dict(zip(uc.tolist(), fr))
    for r in recs:
        if not len(r["cand"]): continue
        peaks = []
        for a_, b_ in r.pop("specs"):
            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
        F1 = np.zeros((len(r["cand"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)
        for j, c in enumerate(r["cand"]):
            fm = fmap[int(c)]
            for k, (x, y) in enumerate(peaks):
                F1[j, k] = pv.explain_score(fm, x, y, mode=r["mode"], tol=CFG.MZ_TOL)
                F2[j, k] = pv.explain_score_adduct(fm, x, y, r["adducts"][k], tol=CFG.MZ_TOL)
                F3[j, k] = pv.explain_score(fm, x, y, mode=r["modes"][k], tol=CFG.MZ_TOL)
        r["frag"] = F1.max(1); r["frag_ad"] = F2.max(1); r["frag_mean"] = F3.mean(1); r["frag_pol"] = F3.max(1)


def _hgb(X, Y, W, seed, gbm):
    from sklearn.ensemble import HistGradientBoostingClassifier
    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)
    return m


def fit_rankers(rank_train_path):
    """Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim)."""
    z = np.load(rank_train_path, allow_pickle=True)
    X, Y, S = z["X"], z["Y"], z["S"]
    X = retie(X, z["G"], RANK_COLS_B)
    log("our ranker: rank columns recomputed with averaged ties")
    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]
    log(f"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features")
    return rankers, [str(b) for b in z["blocks"]]


def fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):
    """prvsiyan's ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim)."""
    z = np.load(pv_rank_path)
    # In these rows a candidate list is (molecule G, simulation M).
    X = retie(z["X"], z["G"] * 2 + z["M"].astype(int), RANK_COLS_BASE)
    log("prvsiyan ranker: rank columns recomputed with averaged ties")
    rankers = [_hgb(X, z["Y"], np.where(z["M"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]
    log(f"prvsiyan ranker: {len(rankers)} GBMs on {z['X'].shape[0]:,} rows x {z['X'].shape[1]} features")
    return rankers, z["X"].shape[1]


def score_molecules(recs, rankers, blocks, use_fp):
    out = {}
    for r in recs:
        if not len(r["cand"]): continue
        rr = r if use_fp else dict(r, zlog=None)
        cfp = pool_fps(r["cand"])
        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)
        out[r["mid"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)
    return out


def blend_scores(a, b, wa):
    """Within-molecule rank blend (both inputs are per-candidate probabilities)."""
    out = {}
    for mid in a:
        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])
        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]
    return out


def make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):
    """score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all."""
    from multiprocessing import Pool as MPool
    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}
    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind="mergesort")][:topn + 15] for mid, sc in S.items()}
               for n, S in score_sets.items()}
    allc = [v for o in ordered.values() for v in o.values()]
    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)
    with MPool(workers) as mp:
        ck = mp.map(canon_key, list(POOL["smiles"][short]), chunksize=64)
    ckey = dict(zip(short.tolist(), ck))
    samp = pd.read_csv(sample_path)
    subs = {}
    for n, o in ordered.items():
        rows = []
        for mid, _ in mols:
            out, seen = [], set()
            for c in o.get(mid, []):
                k = ckey.get(int(c)) or POOL["keys"][c]
                if k in seen: continue
                seen.add(k); out.append(POOL["smiles"][c])
                if len(out) == topn: break
            while len(out) < topn:
                out.append("CCO")
            rows.append((mid, ";".join(out[:topn])))
        sub = samp[["molecule_id"]].merge(pd.DataFrame(rows, columns=["molecule_id", "smiles"]), on="molecule_id", how="left")
        sub["smiles"] = sub["smiles"].fillna("CCO")
        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0
        assert (sub.smiles.str.split(";").map(len) == topn).all()
        subs[n] = sub
    return subs


def main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):
    global POOL
    import torch
    import pv_fp
    L = load_library(train_path); log(f"library {len(L['prec']):,} spectra")
    POOL = build_pool(L, workers)
    L["row_p"] = POOL["k2i"].reindex(L["ik"]).fillna(-1).values.astype(np.int64)
    del L["ik"], L["smi"]
    I = build_index(L)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)
    log(f"fp models: {len(single)} single + {len(merged)} merged on {dev}")
    te = pq.read_table(test_path).to_pandas()
    if limit:
        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]
    log(f"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules")
    mols, recs = compute_channels(L, I, (single, merged, dev), te)
    del L, I
    compute_frag(recs, workers)
    ours, blocks = fit_rankers(our_rank_path)
    s_ours = score_molecules(recs, ours, blocks, use_fp=False)
    del ours
    pvr, nfeat = fit_pv_rankers(pv_rank_path)
    s_pv = score_molecules(recs, pvr, ["base"], use_fp=True)
    del pvr
    subs = make_submissions(mols, recs, {"blend": blend_scores(s_pv, s_ours, w_pv), "pv": s_pv, "ours": s_ours},
                            sample_path, workers)
    log("submissions ready")
    return subs, recs


## 4 · Run
Both rankers are fitted here: ours from `megayak/casmi26-simulated-ranker-rows` (4 HistGradientBoosting models), prvsiyan's from `prvsiyan/casmi26-ranker-features` (8 models: 2 priors × 4 seeds, as in his notebook).

## 3b · Our pipeline (ryqn00)

Our own library search, candidate pool, fingerprint model (v5) and learned ranker
(github docs/16). It runs first, keeps only a per-candidate score per test
molecule, and frees its memory before the engine starts.


In [ ]:
import os, glob, subprocess, sys
# ms_entropy (the Flash Entropy library search) ships as a wheel in casmi26-deps.
deps = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ms_entropy-*.whl', recursive=True)})
assert deps, 'no ms_entropy wheel: is ryqn00/casmi26-deps attached?'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                f'--find-links={deps[0]}', '--no-deps', 'ms_entropy'], check=True)
os.makedirs('casmi', exist_ok=True)   # %%writefile does not create directories


In [ ]:
%%writefile casmi/__init__.py
"""Our pipeline (github: Ryqn0/enveda-casmi-2026), src/casmi."""


In [ ]:
%%writefile casmi/pool.py
"""Candidate pool: every structure a test molecule might plausibly be.

WHY THIS EXISTS
    Phase 1 could only ever answer with a structure that had a training
    spectrum. Competition class 2 is defined as "structure is known, but no
    public spectra exist", so those molecules were unreachable — not badly
    ranked, absent. The leaderboard confirmed it: we scored 0.130, which implies
    class 1 is only ~15% of the test set.

    A candidate pool fixes the retrieval half of that. We take every structure
    from COCONUT (natural products) plus the training set, and for a query we
    keep the ones whose neutral mass matches the measured precursor. Those are
    the molecules it *could* be. Ranking them is a separate problem.

HOW A MASS BECOMES A FILTER
    The instrument measures the ion, not the molecule: an [M+H]+ ion weighs the
    molecule plus one proton. Subtract the adduct's contribution and you have
    the neutral mass, which is a property of the structure alone and can be
    compared against a database.
"""

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# Monoisotopic masses: the mass of each element's MOST ABUNDANT isotope, not the
# periodic-table average. A mass spectrometer resolves individual isotopes, so
# using the average weight (e.g. Cl = 35.45) would be wrong by a third of a Dalton.
MONOISOTOPIC = {
    "C": 12.000000, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221,
    "S": 31.97207069, "P": 30.97376151, "F": 18.9984032, "Cl": 34.96885271,
    "Br": 78.9183376, "I": 126.904468, "Si": 27.9769265327, "Se": 79.9165196,
    "B": 11.0093055, "Na": 22.98976928, "K": 38.96370668, "Li": 7.0160040,
    "Mg": 23.9850419, "Ca": 39.9625912, "Fe": 55.9349421, "Zn": 63.9291466,
    "Cu": 62.9295989, "Mn": 54.9380471, "Co": 58.9331976, "As": 74.9215942,
    "D": 2.0141017778,
}

ELECTRON = 0.00054858

# What each adduct ADDS to the neutral monoisotopic mass to give the measured
# m/z. All are singly charged, so m/z equals mass.
#
# The electron correction matters at the accuracy we need: a proton is a
# hydrogen ATOM minus its electron (1.007825 - 0.000549 = 1.007276). Ignoring
# that is a 1.5 ppm error at m/z 350 — comparable to the tolerance itself.
_H = MONOISOTOPIC["H"]
_H2O = 2 * _H + MONOISOTOPIC["O"]
ADDUCT_MASS = {
    "[M+H]+":        _H - ELECTRON,
    "[M+NH4]+":      MONOISOTOPIC["N"] + 4 * _H - ELECTRON,
    "[M-H2O+H]+":    _H - ELECTRON - _H2O,
    "[M-2H2O+H]+":   _H - ELECTRON - 2 * _H2O,
    "[M+Na]+":       MONOISOTOPIC["Na"] - ELECTRON,
    "[M+K]+":        MONOISOTOPIC["K"] - ELECTRON,
    "[M-H]-":        -_H + ELECTRON,
    "[M-H2O-H]-":    -_H + ELECTRON - _H2O,
    "[M+CH2O2-H]-":  MONOISOTOPIC["C"] + 2 * _H + 2 * MONOISOTOPIC["O"] - _H + ELECTRON,
    "[M+Cl]-":       MONOISOTOPIC["Cl"] + ELECTRON,
}

# Matches one element symbol plus an optional count: "C16" -> ("C", "16"),
# "S" -> ("S", ""). [A-Z][a-z]? is an uppercase letter optionally followed by a
# lowercase one, which is exactly how element symbols are written.
_FORMULA_TOKEN = re.compile(r"([A-Z][a-z]?)(\d*)")


def formula_mass(formula: str) -> float | None:
    """Monoisotopic mass of a molecular formula, or None if it cannot be read.

    Returns None for formulas containing an unknown element, rather than
    silently treating it as weightless — a wrong mass would quietly exclude the
    right candidate from every future search.
    """
    if not isinstance(formula, str) or not formula:
        return None
    # Charge markers and hydrates would corrupt a neutral mass, so reject them.
    if any(c in formula for c in "+-."):
        return None

    total = 0.0
    consumed = 0
    for sym, count in _FORMULA_TOKEN.findall(formula):
        if sym not in MONOISOTOPIC:
            return None
        total += MONOISOTOPIC[sym] * (int(count) if count else 1)
        consumed += len(sym) + len(count)

    # findall skips anything it cannot match, so compare the consumed length
    # against the input to catch stray characters instead of ignoring them.
    if consumed != len(formula):
        return None
    return total


def build_pool(coconut_csv: Path, train_parquet: Path, out: Path) -> pd.DataFrame:
    """Merge COCONUT and the training structures into one mass-indexed pool."""
    # --- COCONUT: the natural-product database ---------------------------
    # Only the four columns we need; the file is 577 MB and has 40 columns.
    coco = pd.read_csv(
        coconut_csv,
        usecols=["canonical_smiles", "standard_inchi_key", "molecular_formula",
                 "exact_molecular_weight"],
    )
    coco = coco.rename(columns={"canonical_smiles": "smiles"})
    coco["inchikey14"] = coco.standard_inchi_key.str.slice(0, 14)
    coco["source"] = "coconut"

    # --- training structures ---------------------------------------------
    # Already in the competition data, and they cover class 1. Included so one
    # pool serves every class.
    tr = pd.read_parquet(
        train_parquet, columns=["inchikey14", "normalized_smiles", "molecular_formula"]
    )
    tr = tr.rename(columns={"normalized_smiles": "smiles"})
    tr = tr.drop_duplicates(subset="inchikey14")
    tr["source"] = "train"
    tr["exact_molecular_weight"] = np.nan

    pool = pd.concat(
        [coco[["inchikey14", "smiles", "molecular_formula", "exact_molecular_weight",
               "source"]], tr],
        ignore_index=True,
    )

    # Drop rows with no usable key or structure before doing expensive work.
    pool = pool[pool.inchikey14.notna() & pool.smiles.notna()]

    # Prefer training entries on collision: their SMILES are RDKit-standardised
    # the same way the competition labels are. keep="last" works because the
    # training rows were concatenated after COCONUT's.
    pool = pool.drop_duplicates(subset="inchikey14", keep="last")

    # Mass from formula, which is exact and fast. Parsing a million SMILES with
    # RDKit would take half an hour and give the same answer.
    mass = pool.molecular_formula.map(formula_mass)
    pool["exact_mass"] = mass
    pool = pool[pool.exact_mass.notna()]

    # Sorting by mass is what lets a query use binary search instead of scanning
    # the whole pool. reset_index makes positions contiguous afterwards.
    pool = pool.sort_values("exact_mass").reset_index(drop=True)

    out.parent.mkdir(parents=True, exist_ok=True)
    pool[["inchikey14", "smiles", "molecular_formula", "exact_mass", "source"]].to_parquet(
        out, index=False
    )
    return pool


class CandidatePool:
    """Mass-sorted structures, queried by precursor m/z and adduct."""

    def __init__(self, path: Path):
        self.df = pd.read_parquet(path)
        # Hot arrays pulled out of pandas once; indexing numpy is far cheaper.
        self.mass = self.df.exact_mass.to_numpy()
        self.smiles = self.df.smiles.to_numpy()
        self.key = self.df.inchikey14.to_numpy()
        assert np.all(np.diff(self.mass) >= 0), "pool must be sorted by mass"

    def __len__(self) -> int:
        return len(self.df)

    def neutral_mass(
        self, precursor_mz: float, adduct: str, offset_ppm: float = 0.0
    ) -> float | None:
        """Undo the adduct: measured ion m/z -> mass of the neutral molecule.

        `offset_ppm` removes a known systematic calibration bias. Measured from
        train.parquet's `precursor_error_ppm`, every source library reads HIGH
        against the mass implied by its own labelled structure:

            enveda-180 (timsTOF)  +0.98 ppm   <- the test instrument
            gnps       (Orbitrap) +1.55 ppm
            massbank   (QTOF)     +1.87 ppm
            riken      (QTOF)     +3.29 ppm

        This is not noise, it is calibration. Correcting it lets the search
        window be CENTRED on the truth rather than straddling it, so a narrower
        window keeps the same recall — and a narrower window means fewer
        candidates competing for 25 slots.
        """
        add = ADDUCT_MASS.get(adduct)
        if add is None:
            return None
        m = precursor_mz - add
        # The measurement is high by offset_ppm, so divide it back down.
        return m / (1.0 + offset_ppm * 1e-6)

    def query(
        self,
        precursor_mz: float,
        adduct: str,
        ppm: float = 10.0,
        offset_ppm: float = 0.0,
    ) -> np.ndarray:
        """Pool positions whose mass matches this measurement within `ppm`.

        Tolerance is in parts-per-million, not Daltons, because instrument mass
        error scales with mass: 5 ppm is 0.0017 Da at m/z 350 but 0.005 Da at
        m/z 1000.
        """
        m = self.neutral_mass(precursor_mz, adduct, offset_ppm)
        if m is None or m <= 0:
            return np.empty(0, dtype=np.int64)
        tol = m * ppm * 1e-6
        # Two binary searches on the sorted mass array bound the window.
        lo, hi = np.searchsorted(self.mass, [m - tol, m + tol])
        return np.arange(lo, hi, dtype=np.int64)


def _self_check() -> None:
    # Caffeine C8H10N4O2, monoisotopic 194.08038
    m = formula_mass("C8H10N4O2")
    assert abs(m - 194.080376) < 1e-4, m
    # Glucose C6H12O6, 180.06339
    assert abs(formula_mass("C6H12O6") - 180.063388) < 1e-4
    # Multi-letter element symbols must not be read as two elements.
    assert abs(formula_mass("CCl4") - (12.0 + 4 * 34.96885271)) < 1e-6
    # Unknown or malformed input must be rejected, never silently zero-weighted.
    assert formula_mass("C6H12Xx2") is None
    assert formula_mass("C6H12O6+") is None   # charged
    assert formula_mass("C6H12O6.H2O") is None  # hydrate
    assert formula_mass("") is None

    # An [M+H]+ ion of caffeine is measured at 195.0877.
    off = ADDUCT_MASS["[M+H]+"]
    assert abs((194.080376 + off) - 195.087652) < 1e-4, 194.080376 + off
    # [M-H]- of glucose is 179.0561.
    assert abs((180.063388 + ADDUCT_MASS["[M-H]-"]) - 179.056112) < 1e-4

    print("pool self-check passed")


if __name__ == "__main__":
    _self_check()


In [ ]:
%%writefile casmi/analog.py
"""Analog propagation: rank candidates by resemblance to a spectral cousin.

THE PROBLEM THIS SOLVES
    A competition class-2 molecule has no reference spectrum anywhere, so
    identity search returns nothing and scores exactly zero. But chemistry is
    not random: an unknown natural product almost always has a *relative* that
    has been measured.

THE CHAIN OF REASONING
    1. Hybrid search finds library spectra that match apart from a constant mass
       shift. A constant shift is the signature of a chemical modification.
    2. Those hits identify a known molecule the unknown resembles — the analog.
    3. The pool gives every structure of the right mass.
    4. Prefer candidates that look chemically like the analog.

    The unknown is never identified directly. It is located by triangulation
    from a known neighbour.
"""

from __future__ import annotations

from functools import lru_cache

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

# Morgan fingerprints (a.k.a. ECFP4 at radius 2). Each bit records the presence
# of one circular atom neighbourhood, so two molecules sharing substructures
# share bits. radius=2 means "each atom plus everything within two bonds",
# which is the standard choice for similarity work.
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# How sharply to favour strong analogs. score**4 means an analog matching at
# 0.9 counts ~2.5x more than one at 0.7, and a weak 0.3 match is suppressed to
# nearly nothing. Without this, a crowd of poor analogs outvotes one good one.
DEFAULT_POWER = 4.0


@lru_cache(maxsize=2_000_000)
def morgan_fp(smiles: str):
    """Fingerprint for a SMILES, or None if RDKit cannot parse it.

    Cached because candidate pools overlap heavily between queries — molecules
    of similar mass keep reappearing — and parsing dominates the cost.
    """
    mol = Chem.MolFromSmiles(smiles)
    return None if mol is None else _GEN.GetFingerprint(mol)


def rank_by_analogy(
    candidate_smiles: np.ndarray,
    analog_smiles: np.ndarray,
    analog_scores: np.ndarray,
    power: float = DEFAULT_POWER,
) -> np.ndarray:
    """Score each candidate by its weighted similarity to the best analog.

    score(candidate) = max over analogs of  analog_similarity**power
                                            * Tanimoto(candidate, analog)

    `max` rather than a sum: one convincing analog is strong evidence, whereas
    summing would reward a candidate that is vaguely similar to many unrelated
    hits. Returns one score per candidate, aligned to `candidate_smiles`.
    """
    scores = np.zeros(len(candidate_smiles), dtype=np.float32)
    if len(candidate_smiles) == 0 or len(analog_smiles) == 0:
        return scores

    # Fingerprint the candidates once, keeping track of which survived parsing.
    # BulkTanimotoSimilarity cannot accept a None in its list.
    fps, keep = [], []
    for i, smi in enumerate(candidate_smiles):
        fp = morgan_fp(smi)
        if fp is not None:
            fps.append(fp)
            keep.append(i)
    if not fps:
        return scores
    keep = np.asarray(keep)

    for a_smi, a_score in zip(analog_smiles, analog_scores):
        # A weak analog cannot lift any candidate above what a strong one
        # already gave, so skip it rather than paying for the comparison.
        weight = float(a_score) ** power
        if weight <= 0:
            continue
        a_fp = morgan_fp(a_smi)
        if a_fp is None:
            continue
        # One C++ call for all candidates instead of a Python loop.
        sims = np.asarray(
            DataStructs.BulkTanimotoSimilarity(a_fp, fps), dtype=np.float32
        )
        # Plain assignment, NOT np.maximum(..., out=scores[keep]): indexing with
        # an integer array returns a copy, so an `out=` write would land in a
        # temporary and be discarded, leaving every score at zero.
        scores[keep] = np.maximum(scores[keep], weight * sims)

    return scores


def _self_check() -> None:
    caffeine = "CN1C=NC2=C1C(=O)N(C)C(=O)N2C"
    theophylline = "Cn1c(=O)c2[nH]cnc2n(C)c1=O"   # caffeine minus one methyl
    glucose = "OCC1OC(O)C(O)C(O)C1O"               # unrelated

    assert morgan_fp("not a molecule") is None
    assert morgan_fp(caffeine) is not None

    # A caffeine-like analog must rank theophylline above glucose.
    cands = np.array([glucose, theophylline])
    s = rank_by_analogy(cands, np.array([caffeine]), np.array([1.0]))
    assert s[1] > s[0], s

    # A weaker analog must produce uniformly weaker scores.
    weak = rank_by_analogy(cands, np.array([caffeine]), np.array([0.5]))
    assert weak[1] < s[1]

    # Unparseable candidates keep their slot at score 0 rather than shifting
    # the alignment of everything after them.
    s2 = rank_by_analogy(
        np.array(["@@bad@@", theophylline]), np.array([caffeine]), np.array([1.0])
    )
    assert s2[0] == 0.0 and s2[1] > 0.0, s2

    # No analogs at all is a valid state (nothing matched); score everything 0.
    assert rank_by_analogy(cands, np.array([]), np.array([])).tolist() == [0.0, 0.0]

    print("analog self-check passed")


if __name__ == "__main__":
    _self_check()


In [ ]:
%%writefile casmi/library.py
"""Spectral library: index the training spectra, then rank structures for a query.

This file is embedded verbatim into the Kaggle submission notebook by
`scripts/package_kaggle.py`, so it is commented for reading, not just for
maintenance. It is the whole of the Phase 1 pipeline.

THE IDEA IN THREE SENTENCES
    A mass spectrum is a set of (fragment mass, intensity) pairs. Two spectra of
    the same molecule look alike, so we can identify an unknown by finding the
    training spectrum it most resembles and returning that molecule's structure.
    "Resembles" is measured by entropy similarity, and the search is made fast by
    an index that only compares spectra whose precursor masses already match.

WHY THIS CANNOT SOLVE THE WHOLE PROBLEM
    It can only ever return a structure that is already in the training set. The
    competition deliberately includes molecules that have no public spectra
    (class 2) or are unknown to chemistry entirely (class 3). Those score zero
    here, by construction, no matter how good the similarity function is.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from ms_entropy import FlashEntropySearch

# --------------------------------------------------------------------------
# Tuning constants
# --------------------------------------------------------------------------

# Keep at most this many peaks per library spectrum. The raw data contains
# spectra with up to 73,318 peaks, nearly all of it electronic noise. Capping
# bounds index memory at roughly 1 KB per spectrum (128 peaks x 2 values x 4
# bytes) and removes noise that would only dilute the similarity score.
MAX_PEAKS = 128

# Discard peaks shorter than 1% of the tallest peak in the same spectrum.
# Detector noise lives down there; real fragments generally do not.
NOISE_THRESHOLD = 0.01

# The REAL test set, taken from the competition's data description page.
# Deliberately NOT measured from the shipped test.parquet: that file is a
# placeholder built out of training rows and is swapped for the hidden test set
# when the notebook is re-run for scoring. Fitting these constants to the
# placeholder (245-460 m/z, 7 adducts) would silently discard most of the real
# test range.
TEST_ADDUCTS = frozenset(
    {
        # An "adduct" is what the molecule picked up to become an electrically
        # charged ion, which is the only form a mass spectrometer can weigh.
        # [M+H]+ means "the molecule plus one proton"; its measured mass is
        # therefore the molecule's mass plus ~1.007.
        "[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
        "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-",
    }
)

# Test molecules weigh 157-1159 Da. Adduct offsets span about -19 to +46, so the
# measured precursor m/z can land anywhere in this padded window.
TEST_MZ_RANGE = (130.0, 1220.0)

# Columns read from train.parquet. Listing them explicitly matters: the two peak
# columns are ~2.7 GB of the 3 GB file, and asking for columns we do not need
# would multiply the memory cost for nothing.
LIB_COLUMNS = [
    "inchikey14",          # 14-char hash of the 2D structure - our identity key
    "normalized_smiles",   # the structure itself, as a text string
    "adduct",              # e.g. "[M+H]+"
    "instrument_type",     # e.g. "timsTOF"
    "precursor_mz",        # measured mass of the whole ionised molecule
    "ms2_mzs",             # array of fragment masses
    "ms2_normalized_intensities",  # array of fragment heights, aligned to above
]


def _top_peaks(mzs, intensities, n: int = MAX_PEAKS) -> np.ndarray:
    """Return the n most intense peaks as an (N, 2) array sorted by mass.

    ms_entropy expects peaks shaped [[mz, intensity], [mz, intensity], ...] in
    ascending mass order, so this both truncates and reorders.
    """
    # float32 halves memory versus float64 and is far finer than the instrument's
    # actual mass accuracy, so no real precision is lost.
    mz = np.asarray(mzs, dtype=np.float32)
    it = np.asarray(intensities, dtype=np.float32)

    if mz.size > n:
        # argpartition finds the n largest intensities WITHOUT sorting the rest.
        # That is O(size) instead of O(size log size) - it matters because this
        # runs two million times. The -n index means "the last n after
        # partitioning", i.e. the n biggest.
        keep = np.argpartition(it, -n)[-n:]
        mz, it = mz[keep], it[keep]

    # argpartition returned them in arbitrary order; restore ascending mass.
    order = np.argsort(mz)

    # Glue the two 1-D arrays into one (N, 2) array: column 0 mass, column 1
    # intensity.
    return np.stack([mz[order], it[order]], axis=1)


def iter_library_rows(
    train_path: Path,
    mz_range: tuple[float, float],
    adducts: set[str],
    batch_size: int = 50_000,
):
    """Stream train.parquet in batches, yielding only the rows worth indexing.

    Streaming rather than reading the file whole is not an optimisation, it is a
    requirement: the file is 3 GB on disk and expands well beyond that in memory,
    and we still have to build a multi-gigabyte index alongside it.
    """
    # ParquetFile opens the file's metadata without reading any row data.
    pf = pq.ParquetFile(train_path)
    lo, hi = mz_range

    # Running count of rows seen so far, used to give every row a stable id.
    offset = 0

    # iter_batches yields chunks instead of the whole table, so peak memory stays
    # flat regardless of file size.
    for batch in pf.iter_batches(batch_size=batch_size, columns=LIB_COLUMNS):
        df = batch.to_pandas()

        # row_id is this row's absolute position in train.parquet. Validation
        # uses it to pull a spectrum's raw peaks back out and to hide that exact
        # measurement from the library. Computed before filtering, so the numbers
        # stay aligned to the real file.
        df["row_id"] = np.arange(offset, offset + len(df), dtype=np.int64)
        offset += len(df)

        # Three filters, combined with & (elementwise AND across the whole
        # column at once - far faster than looping over rows):
        df = df[
            df.precursor_mz.between(lo, hi)     # inside the test mass window
            & df.adduct.isin(adducts)           # an adduct the test set uses
            & df.normalized_smiles.notna()      # has a structure label at all
        ]

        # Skip empty batches so the caller never handles a zero-row frame.
        if len(df):
            yield df


def build(
    train_path: Path,
    out_dir: Path,
    mz_range: tuple[float, float],
    adducts: set[str],
) -> None:
    """Build the searchable index and save it, with metadata, to out_dir."""
    # spectra: what ms_entropy indexes. meta: everything we need to interpret a
    # result (which molecule was it?). They must stay in lockstep.
    spectra, meta = [], []

    for df in iter_library_rows(train_path, mz_range, adducts):
        # itertuples is used rather than iterrows because it is roughly 10x
        # faster - it yields lightweight namedtuples instead of building a pandas
        # Series per row.
        for row in df.itertuples(index=False):
            spectra.append(
                {
                    # "id" is our own field. ms_entropy carries unknown keys
                    # through untouched, which is how we recover the original
                    # ordering below. Its value is simply this spectrum's
                    # position in the list we are building.
                    "id": len(spectra),
                    "precursor_mz": float(row.precursor_mz),
                    "peaks": _top_peaks(row.ms2_mzs, row.ms2_normalized_intensities),
                }
            )
        # Append the whole batch's metadata at once. Row order here matches the
        # order the loop above appended spectra, which is what keeps them aligned.
        meta.append(
            df[["inchikey14", "normalized_smiles", "adduct", "instrument_type", "row_id"]]
        )

    # Stitch the per-batch frames into one table. ignore_index renumbers rows
    # 0..N-1 so positions line up with the spectra list.
    meta = pd.concat(meta, ignore_index=True)
    print(f"  collected {len(spectra):,} spectra")

    fes = FlashEntropySearch()

    # build_index does the heavy lifting: cleans every spectrum, then builds the
    # lookup structures that make searching millions of spectra fast.
    ordered = fes.build_index(
        spectra,
        max_peak_num=MAX_PEAKS,
        noise_threshold=NOISE_THRESHOLD,
        clean_spectra=True,
    )

    # ---------------------------------------------------------------------
    # THE SUBTLE PART. build_index SORTS the spectra by precursor mass and
    # returns them in the new order. It may also drop spectra that end up empty
    # after cleaning. If we left `meta` in its original order, every similarity
    # score would be attributed to the WRONG molecule - and nothing would crash.
    # So we read back each surviving spectrum's "id" (its original position) and
    # reorder meta to match exactly.
    # ---------------------------------------------------------------------
    perm = np.fromiter((s["id"] for s in ordered), dtype=np.int64, count=len(ordered))
    meta = meta.iloc[perm].reset_index(drop=True)

    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "flash").mkdir(exist_ok=True)
    fes.write(out_dir / "flash")  # ms_entropy's own binary format

    # One row per unique structure. Many spectra share a structure (a
    # well-studied molecule can have hundreds), and predictions are per
    # structure, so we need this collapsed view.
    structures = (
        meta.groupby("inchikey14", sort=True)     # sort=True fixes a stable order
        .agg(smiles=("normalized_smiles", "first"))  # any of its SMILES will do
        .reset_index()
    )

    # For each indexed spectrum, which row of `structures` does it belong to?
    # get_indexer looks up every value at once and returns integer positions,
    # which is what lets the scoring step stay pure numpy.
    meta["struct_idx"] = pd.Index(structures.inchikey14).get_indexer(meta.inchikey14)

    # A -1 would mean a spectrum whose structure is missing from the table, which
    # can only happen if the two were built from different data. Fail loudly.
    assert (meta.struct_idx >= 0).all(), "every spectrum must map to a structure"

    meta.to_parquet(out_dir / "meta.parquet", index=False)
    structures.to_parquet(out_dir / "structures.parquet", index=False)
    print(f"  indexed {len(meta):,} spectra / {len(structures):,} structures")


class Library:
    """A loaded index plus the lookup tables needed to rank structures."""

    def __init__(self, out_dir: Path):
        # Construct then read: ms_entropy wants the path at construction time and
        # loads the arrays on .read().
        self.fes = FlashEntropySearch(path_data=out_dir / "flash", low_memory=False)
        self.fes.read(out_dir / "flash")

        self.meta = pd.read_parquet(out_dir / "meta.parquet")
        self.structures = pd.read_parquet(out_dir / "structures.parquet")

        # Pull the hot columns out of pandas into plain numpy arrays once.
        # Indexing a numpy array is far cheaper than indexing a DataFrame, and
        # these are touched on every single query.
        self.struct_idx = self.meta.struct_idx.to_numpy()
        self.n_structures = len(self.structures)
        self.smiles = self.structures.smiles.to_numpy()

        # Group spectra by structure so validation can hide one structure (or one
        # measurement of it) without rebuilding the whole index.
        # Sorting by struct_idx puts every structure's spectra in one contiguous
        # run; searchsorted then records where each run starts. "stable" keeps
        # equal elements in their original relative order, so results are
        # reproducible between runs.
        self._order = np.argsort(self.struct_idx, kind="stable")
        self._starts = np.searchsorted(
            self.struct_idx[self._order], np.arange(self.n_structures + 1)
        )

        # Lookup from train.parquet row_id back to a position in this index.
        self._row_index = pd.Index(self.meta.row_id)

    def spectra_of(self, struct: int) -> np.ndarray:
        """Index positions of every spectrum belonging to one structure."""
        # Slice out this structure's contiguous run set up in __init__.
        return self._order[self._starts[struct] : self._starts[struct + 1]]

    def positions_of_rows(self, row_ids) -> np.ndarray:
        """Index positions for given train.parquet row ids; unknown ids dropped."""
        pos = self._row_index.get_indexer(pd.Index(row_ids))
        return pos[pos >= 0]  # get_indexer returns -1 for "not found"

    def score_spectrum(
        self,
        precursor_mz: float,
        peaks: np.ndarray,
        exclude: np.ndarray | None = None,
    ) -> np.ndarray:
        """Score every structure against ONE query spectrum.

        Returns an array of length n_structures. `exclude` lists index positions
        to ignore; validation uses it to hide the query's own library entry,
        which would otherwise match itself at a perfect 1.0.
        """
        # Apply the same cleaning to the query that the library received, so like
        # is compared with like. This also removes peaks at or near the precursor
        # mass: that peak is the intact molecule that failed to fragment, it is
        # present in every candidate's spectrum, and so it tells us nothing.
        cleaned = self.fes.clean_spectrum_for_search(
            precursor_mz=precursor_mz, peaks=peaks, noise_threshold=NOISE_THRESHOLD
        )

        # A spectrum can be entirely noise and clean away to nothing. Scoring it
        # would crash, so return all-zeros and let the caller carry on.
        if len(cleaned) == 0:
            return np.zeros(self.n_structures, dtype=np.float32)

        # THE ACTUAL SEARCH. "identity" means only library spectra whose
        # precursor mass matches the query's are considered - the same molecule
        # must weigh the same. Returns one similarity per indexed spectrum.
        #   ms1_tolerance: how far the whole molecule's mass may differ (0.01 Da)
        #   ms2_tolerance: how far a fragment's mass may differ (0.02 Da)
        per_spectrum = self.fes.identity_search(
            precursor_mz=precursor_mz,
            peaks=cleaned,
            ms1_tolerance_in_da=0.01,
            ms2_tolerance_in_da=0.02,
        )

        if exclude is not None and len(exclude):
            # Copy first: the array may be a view into ms_entropy's own memory,
            # and writing to it would corrupt the index for later queries.
            per_spectrum = per_spectrum.copy()
            per_spectrum[exclude] = 0.0

        # Collapse per-spectrum scores into per-structure scores.
        out = np.zeros(self.n_structures, dtype=np.float32)

        # np.maximum.at scatters with a maximum: for each spectrum, write its
        # score into its structure's slot, keeping whichever is larger. The ".at"
        # form is essential - a plain out[idx] = vals would keep only the LAST
        # write per structure rather than the best.
        # A structure is judged by its single best-matching spectrum, because
        # averaging would penalise well-studied molecules for also having noisy
        # entries in the library.
        np.maximum.at(out, self.struct_idx, per_spectrum)
        return out

    def hybrid_scores(
        self,
        precursor_mz: float,
        peaks: np.ndarray,
        exclude: np.ndarray | None = None,
    ) -> np.ndarray | None:
        """Best hybrid-search score of EVERY structure (length n_structures).

        None when the query cleans away to nothing. analog_structures keeps the
        top few of these as cousins; the learned ranker also reads a
        candidate's OWN entry: its library spectra matched with a precursor
        shift, i.e. the same molecule measured as a different adduct -- which
        identity search (precursor must match) cannot see.
        """
        cleaned = self.fes.clean_spectrum_for_search(
            precursor_mz=precursor_mz, peaks=peaks, noise_threshold=NOISE_THRESHOLD
        )
        if len(cleaned) == 0:
            return None

        # No ms1_tolerance argument here: allowing the precursor to differ is
        # the entire point of a hybrid search.
        sims = self.fes.hybrid_search(
            precursor_mz=precursor_mz, peaks=cleaned, ms2_tolerance_in_da=0.02
        )
        if exclude is not None and len(exclude):
            sims = sims.copy()
            sims[exclude] = 0.0

        per_struct = np.zeros(self.n_structures, dtype=np.float32)
        np.maximum.at(per_struct, self.struct_idx, sims)
        return per_struct

    def analog_structures(
        self,
        precursor_mz: float,
        peaks: np.ndarray,
        top_n: int = 50,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Find library structures that are chemically RELATED to the query.

        `identity_search` demands the precursor masses match, so it can only
        find the same molecule. `hybrid_search` allows a constant mass shift
        between query and library spectrum, which is exactly what a chemical
        modification looks like: add a methyl group and every fragment
        containing it moves by the same 14 Da while the rest stay put.

        That makes this the tool for competition class 2 — the molecule itself
        has no reference spectrum, but a cousin of it does.

        Returns (structure indices, scores) for the top_n best analogs.
        """
        per_struct = self.hybrid_scores(precursor_mz, peaks, exclude)
        if per_struct is None:
            return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)

        n = min(top_n, self.n_structures)
        top = np.argpartition(-per_struct, n - 1)[:n]
        top = top[np.argsort(-per_struct[top])]
        return top, per_struct[top]

    def rank_molecule(
        self,
        spectra: list[tuple[float, np.ndarray]],
        k: int = 25,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Fuse a molecule's spectra into one ranked candidate list.

        `spectra` is [(precursor_mz, peaks), ...] - all measurements of the SAME
        molecule, typically at different collision energies. The competition
        scores per molecule, not per spectrum, so they must be combined.

        Returns (structure indices, scores), best first, both of length k.
        """
        total = np.zeros(self.n_structures, dtype=np.float32)

        # Sum each spectrum's per-structure scores...
        for mz, peaks in spectra:
            total += self.score_spectrum(mz, peaks, exclude=exclude)

        # ...then divide to get the mean. Using the mean rather than the maximum
        # rewards structures that match at EVERY collision energy over ones that
        # matched strongly at a single energy by chance. max(...,1) guards
        # against a molecule that somehow arrived with no spectra.
        total /= max(len(spectra), 1)

        # Never ask for more candidates than exist.
        k = min(k, self.n_structures)

        # Find the k best without sorting all ~269,000 structures. Negating turns
        # "largest" into "smallest", which is what argpartition finds.
        top = np.argpartition(-total, k - 1)[:k]

        # argpartition guarantees the right k, but in arbitrary order. Rank is the
        # entire metric here, so sort those k properly.
        top = top[np.argsort(-total[top])]

        return top, total[top]


In [ ]:
%%writefile casmi/fpnet.py
"""Spectrum -> molecular fingerprint: a set transformer over peaks.

WHY THIS SHAPE OF MODEL
    A spectrum is an unordered SET of (m/z, intensity) pairs, not a sequence and
    not an image. Transformer self-attention is permutation-equivariant, so it
    fits naturally: no positional encoding, peaks attend to each other, and the
    order they arrive in cannot matter.

WHY PREDICT A FINGERPRINT RATHER THAN A STRUCTURE
    Generating a SMILES string is an open-ended search over a combinatorial
    space. Predicting a fingerprint is a fixed-size multi-label classification
    with millions of labelled examples. The fingerprint is then used to RANK
    candidates that already exist, so the model never has to invent anything.

WHY IT SHOULD BEAT ANALOG PROPAGATION WHERE WE ARE WEAK
    Analogy needs a spectral cousin to exist. A fingerprint model learns the
    general mapping from fragmentation patterns to substructures, so it scores a
    molecule with no relative in the library at all — which is exactly the
    distant-analog bucket where we measured MRR 0.396.
"""

from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


class SinEmb(nn.Module):
    """Sinusoidal embedding of a continuous value, log-spaced.

    A raw m/z of 372.14 is a poor network input: one number spanning three
    orders of magnitude. Projecting it onto sines and cosines of many
    wavelengths gives a smooth, high-dimensional code where nearby masses are
    nearby vectors — the same trick as positional encoding, applied to a
    physical quantity rather than an index.

    Wavelengths are log-spaced from 10^-2 to 10^3.2 Da so the code resolves both
    a 0.01 Da isotope gap and a 1000 Da precursor.
    """

    def __init__(self, dim: int, lo: float = -2.0, hi: float = 3.2):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, torch.linspace(lo, hi, n))
        # register_buffer: saved with the model and moved to GPU with .to(),
        # but not a trainable parameter.
        self.register_buffer("inv", (2 * math.pi) / wav)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)


class Block(nn.Module):
    """Pre-norm transformer block with masked self-attention."""

    def __init__(self, d: int, heads: int, drop: float):
        super().__init__()
        self.h = heads
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d)
        )
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor, pad: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        y = self.n1(x)
        # One projection for queries, keys and values, then split.
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        # pad is True for padding slots; the mask must be True where attention
        # IS allowed, hence the inversion.
        mask = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    """Predicts fingerprint-bit logits from a spectrum plus its metadata."""

    def __init__(self, n_bits: int, n_adducts: int, n_instruments: int = 4,
                 d: int = 256, layers: int = 4, heads: int = 8, drop: float = 0.1,
                 extra_feats: bool = True):
        super().__init__()
        # extra_feats is IN the cfg so the architecture is self-describing.
        # v2 was trained before the engineered features existed; without this
        # its checkpoint cannot be loaded at all, and a checkpoint you cannot
        # load is a comparison you cannot run.
        self.cfg = dict(n_bits=n_bits, n_adducts=n_adducts,
                        n_instruments=n_instruments, d=d, layers=layers,
                        heads=heads, drop=drop, extra_feats=extra_feats)
        self.extra_feats = extra_feats

        self.mz_emb = SinEmb(d)
        # Neutral loss = precursor - fragment. The SAME chemical group falling
        # off two different molecules gives different fragment masses but the
        # same neutral loss, so this view is translation-invariant and exposes
        # structure the raw m/z hides.
        self.nl_emb = SinEmb(d)
        # 2*d for the two sinusoidal embeddings, +3 for the scalar features:
        # raw intensity, log intensity, and fractional position in the molecule.
        self.peak = nn.Linear(2 * d + (3 if extra_feats else 1), d)

        self.prec_emb = SinEmb(d)
        self.adduct = nn.Embedding(n_adducts, d)
        self.instrument = nn.Embedding(n_instruments, d)
        self.glob = nn.Linear(d + 1, d)       # +1 for collision energy

        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        # Mean and max pooling concatenated: mean captures the overall pattern,
        # max captures whether any single diagnostic peak is present.
        self.head = nn.Sequential(
            nn.Linear(2 * d, 1024), nn.GELU(), nn.Dropout(drop),
            nn.Linear(1024, n_bits),
        )

    def forward(self, mz, inten, pad, prec, add_id, ins_id, ce):
        # Neutral losses, floored at 0 — a fragment heavier than its precursor
        # is noise, and a negative loss has no physical meaning.
        nl = (prec[:, None] - mz).clamp(min=0)

        # --- engineered scalar features -------------------------------------
        # 1. log intensity. Peak heights span orders of magnitude and are
        #    heavily skewed toward near-zero noise, so on a linear scale almost
        #    every peak looks identical to the network. log1p spreads the low
        #    end out where most of the peaks actually are.
        log_int = torch.log1p(inten * 1000.0) / 7.0     # ~[0, 1]
        # 2. fractional position in the molecule, mz / precursor in [0, 1].
        #    Absolute mass is already encoded, but this says whether a fragment
        #    is a small piece or nearly the whole molecule — a scale-invariant
        #    view that transfers across molecules of different size.
        rel_mz = (mz / prec[:, None].clamp(min=1.0)).clamp(0, 1)

        feats = [self.mz_emb(mz), self.nl_emb(nl), inten.unsqueeze(-1)]
        if self.extra_feats:
            feats += [log_int.unsqueeze(-1), rel_mz.unsqueeze(-1)]
        p = self.peak(torch.cat(feats, dim=-1))

        # One global token carrying everything that is true of the whole
        # measurement rather than of any single peak.
        g = self.glob(torch.cat([self.prec_emb(prec),
                                 (ce / 100.0).unsqueeze(-1)], dim=-1))
        g = g + self.adduct(add_id) + self.instrument(ins_id)

        x = torch.cat([g.unsqueeze(1), p], dim=1)
        # The global token is never padding, so prepend a False column.
        pad = torch.cat([torch.zeros(len(pad), 1, dtype=torch.bool,
                                     device=pad.device), pad], dim=1)
        for b in self.blocks:
            x = b(x, pad)
        x = self.norm(x)

        # Exclude padding from pooling: -inf before max, and a masked mean.
        m = (~pad).unsqueeze(-1).float()
        mean = (x * m).sum(1) / m.sum(1).clamp(min=1)
        mx = x.masked_fill(pad.unsqueeze(-1), float("-inf")).max(1).values
        return self.head(torch.cat([mean, mx], dim=-1))


def score_candidates(logits: np.ndarray, cand_fp: np.ndarray,
                     platt: tuple | None = None) -> np.ndarray:
    """Rank candidates by the log-likelihood of their fingerprint.

    `logits` is the model's output for one spectrum, `cand_fp` is (n_cand,
    n_bits) of 0/1. The score is

        sum over bits of [ fp*log(p) + (1-fp)*log(1-p) ]

    i.e. how probable this exact bit pattern is under the model. A plain dot
    product would only reward bits the candidate HAS, so a molecule with many
    bits set would win by having more chances to agree; this also penalises
    bits the model expects and the candidate lacks.
    """
    z = logits.astype(np.float64)
    if platt is not None:
        # Per-bit Platt calibration (scripts/calibrate_fpnet.py). Without it the
        # "probabilities" below are not probabilities: pos_weight biases the
        # model toward predicting 1s (precision 0.21 vs recall 0.76), so every
        # bit is over-stated, and by a different amount per bit. Ranking by a
        # likelihood built from miscalibrated terms weights the evidence wrong.
        a, b = platt
        z = a.astype(np.float64) * z + b.astype(np.float64)
    p = 1.0 / (1.0 + np.exp(-z))
    eps = 1e-6
    lp, lq = np.log(p + eps), np.log1p(-p + eps)
    return cand_fp @ lp + (1 - cand_fp) @ lq


def select_peaks(mzs, intensities, max_peaks: int) -> tuple[np.ndarray, np.ndarray]:
    """The most intense `max_peaks` peaks, returned in ascending m/z order.

    THE SINGLE SOURCE OF TRUTH for peak selection. Training and inference used
    to implement this separately, and they disagreed: training took the top N by
    intensity, inference took `peaks[:N]` — which, because arrays are stored
    sorted by m/z, kept the LOWEST-MASS peaks and discarded every informative
    high-mass fragment. 69.8% of test spectra exceed the cap, so most of the
    test set was scored on the wrong half of its own spectrum.

    Nothing crashed and nothing looked wrong. Two implementations of one concept
    will drift; there is now one.
    """
    mz = np.asarray(mzs, dtype=np.float32)
    it = np.asarray(intensities, dtype=np.float32)
    if mz.size > max_peaks:
        # Deterministic: most intense first, and among EXACTLY equal
        # intensities the earlier peak wins (stable sort). np.argpartition
        # used to pick among ties at the cutoff arbitrarily -- and the
        # arbitrary choice can differ between numpy versions, so local and
        # Kaggle results were not guaranteed identical. The in-browser port
        # (web/core.js) uses the same rule; the golden test found the gap.
        keep = np.sort(np.argsort(-it, kind="stable")[:max_peaks])
        mz, it = mz[keep], it[keep]
    order = np.argsort(mz, kind="stable")
    return mz[order], it[order]


def candidate_bits(smiles_list, bit_sel: np.ndarray) -> np.ndarray:
    """(n_cand, n_bits) 0/1 matrix on the SAME bit selection the model used.

    Lives here, beside the model, so inference and validation cannot drift onto
    different bit definitions — every score would then be computed against the
    wrong bits and the channel would look broken for an unrelated reason.
    """
    from rdkit import DataStructs

    from casmi.analog import morgan_fp

    out = np.zeros((len(smiles_list), len(bit_sel)), dtype=np.float64)
    for i, smi in enumerate(smiles_list):
        fp = morgan_fp(smi)
        if fp is None:
            continue
        arr = np.zeros(2048, dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        out[i] = arr[bit_sel]
    return out


def predict_logits(model, spectra, adduct, adducts, instrument_id_, device="cpu",
                   max_peaks: int = 128, collision_energies=None) -> np.ndarray:
    """Mean fingerprint logits over all spectra of one molecule.

    Pooling happens in LOGIT space: these are log-odds, and averaging log-odds
    is the natural way to combine independent evidence about the same bit.

    `collision_energies` must be the REAL per-spectrum values. Training saw a
    median of 32 eV with only 14.5% zeros, so passing 0 — as this function
    silently did until the Phase 3 submission scored 0.196 against Phase 2's
    0.240 — asks the model about an energy it effectively never saw.
    """
    if collision_energies is None:
        # Not a safe default; the median of the training distribution is far
        # less wrong than 0, but callers should pass the real values.
        collision_energies = [32.0] * len(spectra)

    outs = []
    for (mz_val, peaks), ce_val in zip(spectra, collision_energies):
        sel_mz, sel_it = select_peaks(peaks[:, 0], peaks[:, 1], max_peaks)
        n = len(sel_mz)
        mz = np.zeros((1, max_peaks), np.float32)
        it = np.zeros((1, max_peaks), np.float32)
        mz[0, :n], it[0, :n] = sel_mz, sel_it
        pad = np.ones((1, max_peaks), bool)
        pad[0, :n] = False
        with torch.no_grad():
            z = model(
                torch.as_tensor(mz, device=device),
                torch.as_tensor(it, device=device),
                torch.as_tensor(pad, device=device),
                torch.as_tensor([mz_val], dtype=torch.float32, device=device),
                torch.as_tensor([adducts.index(adduct) if adduct in adducts else 0],
                                device=device),
                torch.as_tensor([instrument_id_], device=device),
                torch.as_tensor([float(ce_val)], dtype=torch.float32, device=device),
            )
        outs.append(z[0].float().cpu().numpy())
    return np.mean(outs, axis=0)


def save(model: FPNet, path, **extra) -> None:
    torch.save({"cfg": model.cfg, "state": model.state_dict(), **extra}, path)


def load(path, device="cpu") -> tuple[FPNet, dict]:
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = dict(ck["cfg"])
    # Checkpoints saved before extra_feats existed have no such key. Infer it
    # from the layer that actually changed size rather than guessing.
    if "extra_feats" not in cfg:
        cfg["extra_feats"] = ck["state"]["peak.weight"].shape[1] > 2 * cfg["d"] + 1
    model = FPNet(**cfg).to(device)
    model.load_state_dict(ck["state"])
    model.eval()
    return model, ck


def _self_check() -> None:
    # select_peaks must break intensity ties by position, deterministically --
    # the same rule as web/core.js. Keeps 5, 4, 3 and the FIRST of the tied 1s.
    m_, i_ = select_peaks(np.array([60, 50, 40, 30, 20, 10], np.float32),
                          np.array([5, 1, 3, 1, 1, 4], np.float32), 4)
    assert list(m_) == [10, 40, 50, 60] and list(i_) == [4, 3, 1, 5], (m_, i_)

    # Platt calibration with A=1, B=0 is the identity, so it must reproduce
    # uncalibrated scoring exactly. Catches a transposed or misaligned A/B.
    rng = np.random.default_rng(0)
    lg = rng.normal(size=32).astype(np.float32)
    fpc = (rng.random((5, 32)) > 0.5).astype(np.float32)
    ident = (np.ones(32, np.float32), np.zeros(32, np.float32))
    assert np.allclose(score_candidates(lg, fpc),
                       score_candidates(lg, fpc, platt=ident)), \
        "identity Platt changed the scores"
    # And a real calibration must change them, or it is not being applied.
    shift = (np.full(32, 0.5, np.float32), np.full(32, -1.0, np.float32))
    assert not np.allclose(score_candidates(lg, fpc),
                           score_candidates(lg, fpc, platt=shift)), \
        "platt argument was ignored"

    torch.manual_seed(0)
    B, N, BITS = 3, 16, 64
    # eval(), not train(): dropout is stochastic, so the invariance checks below
    # would compare two different random masks and fail for the wrong reason.
    net = FPNet(n_bits=BITS, n_adducts=10, d=64, layers=2, heads=4).eval()

    mz = torch.rand(B, N) * 400
    it = torch.rand(B, N)
    pad = torch.zeros(B, N, dtype=torch.bool)
    pad[0, 10:] = True          # first spectrum has only 10 real peaks
    prec = torch.rand(B) * 400 + 100
    with torch.no_grad():
        out = net(mz, it, pad, prec, torch.zeros(B, dtype=torch.long),
              torch.zeros(B, dtype=torch.long), torch.full((B,), 30.0))
    assert out.shape == (B, BITS), out.shape
    assert torch.isfinite(out).all(), "non-finite logits"

    # Padding must not change the answer: the padded slots of spectrum 0 hold
    # arbitrary values, and masking should make them irrelevant.
    mz2, it2 = mz.clone(), it.clone()
    mz2[0, 10:] = 999.0
    it2[0, 10:] = 5.0
    out2 = net(mz2, it2, pad, prec, torch.zeros(B, dtype=torch.long),
               torch.zeros(B, dtype=torch.long), torch.full((B,), 30.0))
    assert torch.allclose(out[0], out2[0], atol=1e-5), "padding leaked into output"

    # Permuting real peaks must not change the answer either — a spectrum is a
    # set, and the architecture is only correct if it is permutation-invariant.
    perm = torch.randperm(N)
    out3 = net(mz[1:2, perm], it[1:2, perm], pad[1:2, perm], prec[1:2],
               torch.zeros(1, dtype=torch.long), torch.zeros(1, dtype=torch.long),
               torch.full((1,), 30.0))
    assert torch.allclose(out[1], out3[0], atol=1e-5), "not permutation invariant"

    # score_candidates must prefer the candidate matching the model's beliefs.
    logits = np.array([5.0, -5.0, 5.0, -5.0], dtype=np.float32)
    cands = np.array([[1, 0, 1, 0], [0, 1, 0, 1]], dtype=np.float64)
    s = score_candidates(logits, cands)
    assert s[0] > s[1], s

    print("fpnet self-check passed")


if __name__ == "__main__":
    _self_check()


In [ ]:
%%writefile casmi/fragment.py
"""In-silico fragmentation: score a candidate by how much of the spectrum its
own bonds can explain.

WHY THIS CHANNEL EXISTS
    Our other two channels both need a RELATIVE in the spectral library:

      library search   needs the molecule itself to have a reference spectrum
      analog search    needs something structurally similar to have one

    Class 3 molecules (in no database at all) have neither, so both channels
    score every class-3 candidate on essentially nothing. This channel needs no
    library: it takes the candidate structure, breaks its bonds, and asks
    whether the resulting pieces have the masses we actually observed.

    That makes it the only signal we have that works when the molecule is
    genuinely novel -- which is where the leaderboard gap is.

THE IDEA
    A molecule in a collision cell breaks at its weakest bonds. If a candidate
    is correct, most of the observed peaks should correspond to connected
    sub-pieces of it. If it is wrong, the masses will not line up.

    So: enumerate sub-fragments, compute their exact masses, and measure the
    fraction of total peak intensity that some fragment explains.

DELIBERATE SIMPLIFICATIONS
    Real MetFrag does a bond-energy-weighted tree search over multiple bond
    breakages. This breaks at most `max_breaks` single bonds at once, which
    covers the large majority of real fragmentations at a small fraction of the
    cost. By default only ACYCLIC bonds break; RING_BREAKS additionally allows
    cross-ring cleavage of aliphatic rings (two ring bonds at once), which is
    how sugars and saturated cores fragment. Aromatic rings are never opened.

    ponytail: depth-limited breaking, no bond energies. Upgrade to a scored
    tree search if this channel earns more of its place.
"""

from __future__ import annotations

from functools import lru_cache
from itertools import combinations

import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.Descriptors import ExactMolWt

# RDKit complains loudly about candidate structures it dislikes. We already
# handle failures by returning 0.0, so the log noise is pure cost -- and on
# Kaggle it can produce hundreds of MB of output.
RDLogger.DisableLog("rdApp.*")

# Masses that convert a neutral fragment into the ion we actually detect.
PROTON = 1.007276466      # H+ : an H nucleus, i.e. H minus one electron
HYDROGEN = 1.007825032    # a whole neutral H atom, for rearrangement counting

# Collision-induced dissociation routinely moves a hydrogen from one piece to
# the other as the bond breaks, so a fragment is commonly observed at +/- 1 H
# from its nominal mass. Allowing a 1-H window is standard practice; allowing
# more would match almost anything and destroy the score's discriminating power.
H_SHIFTS = (-1, 0, 1)

# Whether to also try cross-ring cleavage of aliphatic rings. Off until the
# honest holdout says it helps -- see _fragment_masses.
RING_BREAKS = False


@lru_cache(maxsize=200_000)
def _fragment_masses(smiles: str, max_breaks: int = 2,
                     rings: bool = False) -> tuple[float, ...]:
    """Every neutral sub-fragment mass this structure can produce.

    Cached because the SAME candidate appears for many different query
    spectra -- the pool is shared across the whole test set, so without the
    cache the identical molecule is re-fragmented hundreds of times.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ()

    # The intact molecule is always a valid "fragment": the precursor ion
    # itself is usually present, and some spectra are dominated by it.
    masses = {ExactMolWt(mol)}

    # Only single bonds that are NOT in a ring. Breaking one ring bond leaves
    # the molecule in one connected piece, so it yields no new fragment mass --
    # it would just cost time and return the parent again.
    breakable = [
        b.GetIdx()
        for b in mol.GetBonds()
        if b.GetBondType() == Chem.BondType.SINGLE and not b.IsInRing()
    ]
    if rings:
        # Cross-ring cleavage: two single bonds of the same ALIPHATIC ring
        # breaking together cut the ring into two pieces. That is how sugars
        # and saturated terpenoid cores fragment, and acyclic-only breaking
        # cannot produce those masses at all. Aromatic bonds are excluded
        # (RDKit types them AROMATIC, not SINGLE): aromatic rings rarely open
        # under collision-induced dissociation.
        breakable += [
            b.GetIdx()
            for b in mol.GetBonds()
            if b.GetBondType() == Chem.BondType.SINGLE and b.IsInRing()
        ]
    if not breakable:
        return tuple(sorted(masses))

    # Try breaking 1 bond, then 2 at once, etc. Breaking n bonds simultaneously
    # is what produces internal fragments (pieces with both ends cut off).
    for n in range(1, max_breaks + 1):
        for combo in combinations(breakable, n):
            try:
                # addDummies=False: we want the real masses of the pieces, not
                # pieces decorated with placeholder atoms that would add mass.
                frag = Chem.FragmentOnBonds(mol, list(combo), addDummies=False)
                pieces = Chem.GetMolFrags(frag, asMols=True, sanitizeFrags=False)
            except Exception:
                # A candidate RDKit cannot fragment is not worth failing the
                # whole run over; it simply contributes no fragment masses.
                continue
            for p in pieces:
                try:
                    masses.add(ExactMolWt(p))
                except Exception:
                    continue

    return tuple(sorted(masses))


def explained_intensity(smiles: str, mzs, intensities, positive: bool,
                        ppm: float = 10.0, max_breaks: int = 2,
                        rings: bool | None = None) -> float:
    """Fraction of total peak intensity this structure can account for.

    Returns a value in [0, 1]. A correct structure typically explains most of
    the intensity; an incorrect isomer explains noticeably less.

    `ppm` is deliberately looser than the 4 ppm used for precursor matching.
    Precursor m/z is measured precisely, but fragment assignment also absorbs
    the error of our simplified chemistry (which H ended up on which piece), so
    a tight window would reject correct assignments.
    """
    mz = np.asarray(mzs, dtype=np.float64)
    inten = np.asarray(intensities, dtype=np.float64)
    total = inten.sum()
    if total <= 0 or mz.size == 0:
        return 0.0

    frags = _fragment_masses(smiles, max_breaks,
                             RING_BREAKS if rings is None else rings)
    if not frags:
        return 0.0

    # Convert neutral fragment masses into the m/z values we would SEE.
    # Positive mode adds a proton, negative mode removes one; the H_SHIFTS
    # then cover hydrogen rearrangement during the break.
    sign = 1.0 if positive else -1.0
    base = np.asarray(frags, dtype=np.float64) + sign * PROTON
    predicted = np.concatenate([base + k * HYDROGEN for k in H_SHIFTS])
    predicted.sort()

    # For each observed peak, find the nearest predicted m/z. searchsorted gives
    # the insertion point, so the nearest value is at that index or the one
    # before it -- checking both is what makes this O(log n) instead of an
    # O(peaks x fragments) full comparison.
    pos = np.searchsorted(predicted, mz)
    lo = predicted[np.clip(pos - 1, 0, len(predicted) - 1)]
    hi = predicted[np.clip(pos, 0, len(predicted) - 1)]
    nearest = np.where(np.abs(mz - lo) < np.abs(mz - hi), lo, hi)

    # Tolerance in ppm scales with m/z: a fixed Da window would be far too
    # tight at low mass and far too loose at high mass.
    matched = np.abs(mz - nearest) <= mz * ppm * 1e-6
    return float(inten[matched].sum() / total)


def score_candidates_frag(smiles_list, mzs, intensities, adduct: str,
                          ppm: float = 10.0, max_breaks: int = 2,
                          rings: bool | None = None) -> np.ndarray:
    """Explained-intensity score for every candidate, as an array."""
    positive = not adduct.rstrip().endswith("-")
    return np.array(
        [explained_intensity(s, mzs, intensities, positive, ppm, max_breaks,
                             rings)
         for s in smiles_list],
        dtype=np.float32,
    )


def score_spectra_frag(smiles_list, spectra, adduct: str,
                       ppm: float = 10.0, max_breaks: int = 2,
                       rings: bool | None = None) -> np.ndarray:
    """Fragment score for a molecule measured as SEVERAL spectra.

    THE SINGLE SOURCE OF TRUTH for which spectrum the fragment channel uses --
    shared by the ranker, the validator and the notebook. Deciding this in each
    caller is how this project's train/serve skew bugs happened.

    Uses the richest spectrum (most peaks) rather than averaging all of them.
    The score is an explained-intensity FRACTION, and fractions over spectra
    of very different richness are not comparable: a 3-peak spectrum is
    trivially "fully explained" by almost any candidate, so averaging it in
    would drag every candidate toward the same score.

    `spectra` is a list of (precursor_mz, peaks) with peaks an (n, 2) array of
    (m/z, intensity) -- the same shape CombinedRanker.rank receives.
    """
    _, peaks = max(spectra, key=lambda s: len(s[1]))
    pk = np.asarray(peaks, dtype=np.float64)
    if pk.ndim != 2 or len(pk) == 0:
        return np.zeros(len(smiles_list), dtype=np.float32)
    return score_candidates_frag(smiles_list, pk[:, 0], pk[:, 1], adduct,
                                 ppm, max_breaks, rings)


def _self_check() -> None:
    # Caffeine, and a peak list built from masses its own bonds can produce.
    caffeine = "Cn1c(=O)c2c(ncn2C)n(C)c1=O"
    frags = _fragment_masses(caffeine)
    assert len(frags) > 1, "no fragments enumerated for caffeine"

    # Build a synthetic spectrum FROM caffeine's own fragments, so a correct
    # structure must explain ~all of it.
    peaks = np.array([f + PROTON for f in frags[:6]])
    inten = np.ones(len(peaks))
    hit = explained_intensity(caffeine, peaks, inten, positive=True)
    assert hit > 0.99, f"caffeine failed to explain its own fragments: {hit}"

    # A structure of a totally different mass must explain much less of it.
    other = explained_intensity("CCCCCCCCCCCCCCCCCC(=O)O", peaks, inten, True)
    assert other < hit, f"unrelated molecule scored as well as the truth: {other}"

    # Empty and unparseable inputs must return 0.0, not raise: the pool
    # contains structures RDKit refuses, and one of them must not kill a run.
    assert explained_intensity(caffeine, [], [], True) == 0.0
    assert explained_intensity("not a smiles", peaks, inten, True) == 0.0

    # Negative mode must shift the other way, so a spectrum built for positive
    # mode should NOT score identically in negative mode.
    neg = explained_intensity(caffeine, peaks, inten, positive=False)
    assert neg < hit, "adduct polarity is being ignored"

    # Cross-ring cleavage: a glucose-like ring gains fragments; benzene-only
    # structures (aromatic bonds are not SINGLE) must be unaffected.
    sugar = "OCC1OC(O)C(O)C(O)C1O"
    assert len(_fragment_masses(sugar, 2, True)) > len(_fragment_masses(sugar, 2, False)), \
        "ring breaking added no fragments to an aliphatic ring"
    tol = "Cc1ccccc1"
    assert _fragment_masses(tol, 2, True) == _fragment_masses(tol, 2, False), \
        "ring breaking touched an aromatic ring"

    print(f"fragment self-check passed ({len(frags)} caffeine fragments)")


if __name__ == "__main__":
    _self_check()


In [ ]:
%%writefile casmi/ranker.py
"""A learned ranker that combines the three evidence channels per molecule.

WHY
    The fixed blend `fpnet + 0.5*analog + 0.25*frag` weighs the channels the
    same way for every molecule. But how much each channel deserves to be
    trusted changes from molecule to molecule:

      - a near-perfect analog match (class 1) should dominate everything
      - a weak best analog means the analog channel is mostly noise here,
        so fpnet and fragments should carry the decision
      - a sparse spectrum explains itself trivially, so fragments mean less

    A small neural network that sees both the channel scores AND this context
    can learn those "trust X when Y" rules from data instead of us guessing
    one weight that has to serve every case.

HOW IT IS TRAINED (scripts/train_ranker.py)
    Listwise: for each molecule, softmax over its candidates' scores and
    maximise the probability of the true one. That directly rewards putting
    the right answer FIRST, which is what MRR mostly measures.

    Trained only on molecules the fingerprint model never saw (docs/09), or it
    would learn to over-trust a channel that was memorising.

THE ONE RULE
    `ranker_features` is the ONLY place features are computed. Training and
    the Kaggle notebook both call it. This project's worst bugs were all the
    same bug: the same quantity computed two ways in two places.
"""

from __future__ import annotations

import numpy as np
import torch
from torch import nn

# Order matters: it is the column order of the feature matrix, and a saved
# model is only valid for the exact order it was trained with. It is stored in
# the checkpoint and checked on load.
FEATURES = [
    "z_fp", "z_an", "z_fr",        # the three channels, per-molecule z-scores
    "rk_fp", "rk_an", "rk_fr",     # the candidate's rank in each channel, 0 = top
    "an_raw", "fr_raw",            # absolute strength, which z-scoring discards
    "fp_gap",                      # fpnet log-likelihood minus the best one
    "an_conf",                     # how good the best spectral relative is
    "log_ncand",                   # how crowded the candidate list is
    "log_npk",                     # how rich the richest spectrum is
]


def _z(x: np.ndarray) -> np.ndarray:
    """Per-molecule z-score; all-equal inputs carry no ranking information."""
    s = x.std()
    return (x - x.mean()) / s if s > 1e-12 else np.zeros_like(x)


def _rank01(x: np.ndarray) -> np.ndarray:
    """Position of each candidate when sorted best-first, scaled to [0, 1).

    kind="stable" so exact ties always break the same way. The default sort
    is not stable, which would let the same inputs produce different features
    on different runs.
    """
    order = np.argsort(-x, kind="stable")
    r = np.empty(len(x), dtype=np.float64)
    r[order] = np.arange(len(x))
    return r / len(x)


def ranker_features(s_fpnet, s_analog, s_frag, an_conf: float,
                    n_peaks: int) -> np.ndarray:
    """(n_candidates, len(FEATURES)) float32 matrix for ONE molecule.

    Every input is available at test time. In particular nothing here may use
    the true answer -- the validator computes one such quantity (Tanimoto of
    the TRUE structure to the analogs) for reporting, and passing it in would
    hand the ranker the label.
    """
    fp = np.asarray(s_fpnet, dtype=np.float64)
    an = np.asarray(s_analog, dtype=np.float64)
    fr = np.asarray(s_frag, dtype=np.float64)
    n = len(fp)
    full = lambda v: np.full(n, v, dtype=np.float64)   # a per-molecule constant
    cols = [
        _z(fp), _z(an), _z(fr),
        _rank01(fp), _rank01(an), _rank01(fr),
        an, fr,
        fp - fp.max(),
        full(an_conf),
        full(np.log(n)),
        full(np.log(max(n_peaks, 1))),
    ]
    assert len(cols) == len(FEATURES), "FEATURES and ranker_features disagree"
    return np.stack(cols, axis=1).astype(np.float32)


class RankNet(nn.Module):
    """Scores one candidate from its feature row. hidden=0 gives a linear model.

    Deliberately tiny (~1.5k parameters at hidden=32). There are only ~2k
    training molecules; a large network would memorise them.
    """

    def __init__(self, n_features: int, hidden: int = 32):
        super().__init__()
        self.cfg = dict(n_features=n_features, hidden=hidden)
        if hidden == 0:
            self.net = nn.Linear(n_features, 1)
        else:
            self.net = nn.Sequential(
                nn.Linear(n_features, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, 1),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def expected_rr(scores, truth: int, k: int = 25) -> float:
    """Reciprocal rank of candidate `truth`, averaged over random tie-breaking.

    "1 + number of candidates scored strictly higher" puts the truth FIRST
    among everything it ties with. Ties are common (many candidates score
    exactly 0 on analog similarity), so that silently favours tie-heavy methods
    such as the fixed blend over continuous ones such as a learned ranker. On
    the leaderboard a tie's order is arbitrary, so average over every position
    the truth could land in among its ties.
    """
    s = np.asarray(scores, dtype=np.float64)
    higher = int((s > s[truth]).sum())
    ties = int((s == s[truth]).sum()) - 1        # other candidates tied with it
    ranks = np.arange(higher + 1, higher + ties + 2)
    return float(np.where(ranks <= k, 1.0 / ranks, 0.0).mean())


def load_ranker(path, device="cpu"):
    """Load a trained ranker plus the standardisation it was trained with."""
    ck = torch.load(path, map_location=device, weights_only=False)
    # A model trained on a different feature list would silently read the
    # wrong columns. Refuse instead.
    assert list(ck["features"]) == FEATURES, (
        f"ranker was trained on {list(ck['features'])}, code has {FEATURES}")
    net = RankNet(**ck["cfg"]).to(device)
    net.load_state_dict(ck["state"])
    net.eval()
    return net, ck


def score_with_ranker(net, ck, feats: np.ndarray) -> np.ndarray:
    """Higher = better. Applies the SAME standardisation used in training."""
    x = (feats - ck["mean"]) / ck["std"]
    with torch.no_grad():
        return net(torch.as_tensor(x, dtype=torch.float32)).numpy()


def _self_check() -> None:
    rng = np.random.default_rng(0)
    fp, an, fr = rng.normal(size=20), rng.random(20), rng.random(20)
    f = ranker_features(fp, an, fr, an_conf=0.7, n_peaks=50)
    assert f.shape == (20, len(FEATURES))
    assert np.isfinite(f).all()

    # Shifting the fpnet scores by a constant must not change anything: the
    # log-likelihood's absolute level is arbitrary, only differences carry
    # information. This is also what lets training rebuild these features
    # from the dumped fp_gap column.
    g = ranker_features(fp + 123.0, an, fr, an_conf=0.7, n_peaks=50)
    assert np.allclose(f, g, atol=1e-5), "features depend on fpnet's offset"

    # Rank 0 must be the best candidate in each channel.
    assert f[np.argmax(fp), FEATURES.index("rk_fp")] == 0.0

    # A single candidate must not produce NaN (std = 0).
    one = ranker_features([1.0], [0.5], [0.2], an_conf=0.1, n_peaks=3)
    assert np.isfinite(one).all()

    print("ranker self-check passed")


if __name__ == "__main__":
    _self_check()


In [ ]:
%%writefile casmi/rank.py
"""Rank a molecule's candidates by combining three evidence channels.

  analog    does the spectrum resemble a library spectrum of something
            structurally similar? (hybrid search + Tanimoto propagation)
  fpnet     do the candidate's substructures match what the model predicts
            from the peaks?
  fragment  can the candidate's own bond breaks explain the observed peaks?
            (needs no library at all -- the only channel for class 3)

By default the channels are combined by a fixed blend of z-scores,
`fpnet + FPNET_ANALOG_WEIGHT * analog + FRAG_WEIGHT * fragment`, with the
fragment weight currently 0 (no measurable gain on natural products). A small
learned ranker (casmi/ranker.py) can replace the blend, but is not used: on
natural products it did not beat the blend (docs/12).

Identity search (exact library match) is still computed by score_components
for analysis, but not used for ranking: hybrid search subsumes it, since a
zero-mass-shift hybrid match IS an identity match.
"""

from __future__ import annotations

import numpy as np

from casmi.analog import rank_by_analogy
from casmi.fpnet import candidate_bits, predict_logits, score_candidates
from casmi.fragment import score_spectra_frag
from casmi.ranker import ranker_features, score_with_ranker
from casmi.library import Library
from casmi.pool import ADDUCT_MASS, CandidatePool

# Blend weights, measured on the NATURAL-PRODUCT holdout (docs/12): 984
# molecules in COCONUT that fpnet v4c never trained on, from the reference
# libraries, ties broken fairly. This is the only holdout found so far that
# reproduces the leaderboard's ordering of past submissions; the earlier
# "timsTOF holdout" was 99.9% enveda-180, a synthetic screening library, and
# ranked them backwards.
#
#   fpnet v4c +              MRR
#   0.00*analog only      0.5628   (analog alone)
#   0.50*analog           0.5865   <- chosen: the LB-best weight (0.244)
#   0.75*analog           0.5919
#   1.00*analog           0.5897
#   1.50*analog           0.5853
#
# 0.5-1.0 is a plateau (differences <= 0.006, within noise), so the weight is
# held at the value the leaderboard already confirmed, and the next submission
# changes ONE thing -- the model -- versus the 0.244 configuration.
FPNET_ANALOG_WEIGHT = 0.5
# In-silico fragmentation (casmi/fragment.py). Off: on natural products it adds
# nothing measurable (+/- 0.001 at every weight tried), and it costs runtime.
# It helped on enveda-180 only (docs/10), which is not the test chemistry.
FRAG_WEIGHT = 0.0


def _z(x: np.ndarray) -> np.ndarray:
    """Z-score, so two differently-scaled channels can be added."""
    s = x.std()
    return (x - x.mean()) / s if s > 1e-12 else np.zeros_like(x)

# Chosen so a confident identity match (~1.0) cannot be displaced by the best
# realistic analog score (~0.5 * 0.5 = 0.25). Validated by checking that class-1
# MRR does not move when the analog channel is switched on.
ANALOG_WEIGHT = 0.5

# Swept on a 200-molecule timsTOF holdout (scripts/validate_class2.py):
#   5 ppm, no offset : 64 candidates, 100.0% recall, MRR 0.7121
#   4 ppm, +0.98 ppm : 57 candidates, 100.0% recall, MRR 0.7163  <- chosen
#   3 ppm, +0.98 ppm : 52 candidates,  99.5% recall, MRR 0.7165
# 4 ppm keeps full recall while cutting candidates 11%; 3 ppm trades 0.5% of
# recall for no measurable gain, so it is not worth the risk on unseen data.
TOP_ANALOGS = 100   # 100 beat 50 by +0.012 on the calibrated metric
PPM = 4.0
# timsTOF reads +0.98 ppm high against the mass implied by the labelled
# structure. Correcting it centres the search window instead of straddling.
MZ_OFFSET_PPM = 0.98
K = 25


class CombinedRanker:
    """Ranks pool candidates by combining the evidence channels (see module docstring)."""

    def __init__(self, lib: Library, pool: CandidatePool,
                 analog_weight: float = ANALOG_WEIGHT,
                 offset_ppm: float = MZ_OFFSET_PPM,
                 fpnet=None, fpnet_ck=None, device: str = "cpu",
                 use_platt: bool = False, ranker=None, ranker_ck=None):
        self.lib = lib
        self.pool = pool
        self.analog_weight = analog_weight
        # Optional so the pipeline still runs without a trained model.
        self.fpnet = fpnet
        self.device = device
        # Optional learned ranker (casmi/ranker.py). When absent, the fixed
        # blend below is used -- which is also the fallback if it misbehaves.
        self.ranker, self.ranker_ck = ranker, ranker_ck
        if fpnet is not None:
            self.bit_sel = np.asarray(fpnet_ck["bit_sel"])
            self.adducts = list(fpnet_ck["adduct_list"])
            # The cap the model was TRAINED at. Guessing 128 for a 256-peak
            # model silently truncates every rich spectrum, and 69.8% of test
            # spectra exceed 128 peaks.
            self.max_peaks = int(fpnet_ck.get("max_peaks", 128))
            # Per-bit Platt calibration (CSI:FingerID / MSNovelist). The
            # checkpoint CARRIES it but it is OFF by default, because measured
            # on the timsTOF holdout it is a large loss:
            #     v3 fpnet alone   raw 0.8147 | A-only 0.8095 | full 0.6409(v2)
            #     v3 +0.25*analog  raw 0.8509 | A-only 0.8461
            # It improves the probabilities a lot (log-loss 0.2187 -> 0.0732)
            # and the RANKING not at all. Ranking reduces to cand_fp @ (A*z+B),
            # so B ~ -3.2 acts as a -3.2 * (bits set) complexity penalty: a
            # prior about molecules, not evidence from the spectrum.
            # Better probabilities != better ranking. Left wired for the record.
            a, b = fpnet_ck.get("platt_a"), fpnet_ck.get("platt_b")
            self.platt = ((np.asarray(a), np.asarray(b))
                          if use_platt and a is not None else None)
        # Held on the instance rather than threaded through every call: it is a
        # property of the instrument, constant for a whole run.
        self.offset_ppm = offset_ppm

        # The two channels live in different index spaces: identity scores are
        # per LIBRARY structure, candidates are per POOL row. InChIKey14 is the
        # only thing they share, so build the bridge once rather than per query.
        self._lib_row_of_key = {
            k: i for i, k in enumerate(lib.structures.inchikey14.to_numpy())
        }

    def score_components(
        self,
        spectra: list[tuple[float, np.ndarray]],
        adduct: str,
        ppm: float = PPM,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
        """Return (pool positions, identity scores, analog scores, an_conf).

        an_conf is the hybrid-search score of the single best spectral
        relative: how much the analog channel deserves to be trusted for THIS
        molecule. The learned ranker uses it; the fixed blend ignores it.

        Kept separate from `rank` so validation can sweep the blend weight
        without paying for the spectral searches again — the analog channel is
        ~40x slower than identity and dominates the cost.
        """
        mz = spectra[0][0]
        cand = (
            self.pool.query(mz, adduct, ppm=ppm, offset_ppm=self.offset_ppm)
            if adduct in ADDUCT_MASS
            else np.empty(0, np.int64)
        )
        empty = np.empty(0, dtype=np.float32)
        if len(cand) == 0:
            return cand, empty, empty, 0.0

        # --- channel 1: identity ------------------------------------------
        # Mean across the molecule's spectra, matching Phase 1.
        ident_struct = np.zeros(self.lib.n_structures, dtype=np.float32)
        for q_mz, peaks in spectra:
            ident_struct += self.lib.score_spectrum(q_mz, peaks, exclude=exclude)
        ident_struct /= len(spectra)

        # Project library-structure scores onto the candidate list. A candidate
        # absent from the library simply scores 0 here, which is correct: no
        # reference spectrum means no identity evidence.
        ident = np.zeros(len(cand), dtype=np.float32)
        for i, key in enumerate(self.pool.key[cand]):
            row = self._lib_row_of_key.get(key)
            if row is not None:
                ident[i] = ident_struct[row]

        # --- channel 2: analog --------------------------------------------
        best = np.zeros(self.lib.n_structures, dtype=np.float32)
        for q_mz, peaks in spectra:
            idx, sc = self.lib.analog_structures(
                q_mz, peaks, top_n=TOP_ANALOGS, exclude=exclude
            )
            np.maximum.at(best, idx, sc)
        n = min(TOP_ANALOGS, self.lib.n_structures)
        a_idx = np.argpartition(-best, n - 1)[:n]
        a_idx = a_idx[np.argsort(-best[a_idx])]

        analog = rank_by_analogy(
            self.pool.smiles[cand], self.lib.smiles[a_idx], best[a_idx]
        )
        return cand, ident, analog, float(best[a_idx[0]])

    def rank(
        self,
        spectra: list[tuple[float, np.ndarray]],
        adduct: str,
        k: int = K,
        ppm: float = PPM,
        exclude: np.ndarray | None = None,
        collision_energies: list[float] | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Return (pool positions, scores) for the k best candidates.

        With a model loaded, ranking is `fpnet + 0.5*analog` (see
        FPNET_ANALOG_WEIGHT). Without one it falls back to analog-only, which
        is the configuration that scored 0.240.

        `collision_energies` must be the real per-spectrum values; the model was
        trained with them and defaults to the training median otherwise.

        The analog-only choice was itself measured, on a 196-molecule
        natural-product holdout (scripts/validate_combined.py):

            strategy        class 1   class 2   projected LB
            identity only    0.8765    0.0921         0.1746
            linear w=4       0.8994    0.5320         0.3673
            gated t=0.7      0.8986    0.5977         0.3954
            analog only      0.8955    0.6207         0.4048   <-

        Hybrid search subsumes identity search: a zero-mass-shift hybrid match
        IS an identity match, and a molecule's Tanimoto to itself is 1.0. So
        identity adds almost nothing on class 1 (0.8986 -> 0.8955, a rounding
        error) while costing a great deal on class 2, where the true answer has
        no spectrum and scores 0 but wrong candidates that happen to be in the
        library score above it.

        `self.analog_weight` is retained so score_components can be re-tested if
        a future channel changes the balance, but it no longer affects ranking.
        """
        cand, ident, analog, an_conf = self.score_components(
            spectra, adduct, ppm, exclude)
        if len(cand) == 0:
            return cand, np.empty(0, dtype=np.float32)

        if self.fpnet is None:
            total = analog
        else:
            # Weights and the measurements behind them: see FPNET_ANALOG_WEIGHT.
            logits = predict_logits(
                self.fpnet, spectra, adduct, self.adducts,
                instrument_id_=0,   # the test set is 100% timsTOF
                device=self.device,
                max_peaks=self.max_peaks,
                collision_energies=collision_energies,
            )
            fp_score = score_candidates(
                logits, candidate_bits(self.pool.smiles[cand], self.bit_sel),
                platt=self.platt,
            )
            # Fragmentation is the slowest channel; skip it when nothing uses it.
            needs_frag = self.ranker is not None or FRAG_WEIGHT != 0
            frag = (score_spectra_frag(self.pool.smiles[cand], spectra, adduct)
                    if needs_frag else np.zeros(len(cand), dtype=np.float32))
            if self.ranker is not None:
                # Same function, same inputs as the training dump -- n_peaks is
                # the RAW peak count of the richest spectrum, as in validation.
                feats = ranker_features(
                    fp_score, analog, frag, an_conf,
                    n_peaks=max(len(pk) for _, pk in spectra))
                total = score_with_ranker(self.ranker, self.ranker_ck, feats)
            else:
                total = (_z(fp_score) + FPNET_ANALOG_WEIGHT * _z(analog)
                         + FRAG_WEIGHT * _z(frag))

        n = min(k, len(cand))
        top = np.argpartition(-total, n - 1)[:n]
        top = top[np.argsort(-total[top])]
        out, scores = cand[top], total[top]

        if len(out) < k:
            out, scores = self._pad(out, scores, spectra[0][0], adduct, ppm, k)
        return out, scores

    def _pad(self, out, scores, mz, adduct, ppm, k):
        """Fill unused slots with the next mass-nearest structures.

        A tight mass window can return fewer than k candidates — sometimes as
        few as 3. The metric charges nothing for a wrong guess beyond the rank
        it occupies, so leaving 22 slots empty forfeits 22 free chances. Widen
        the tolerance and append by mass proximity.

        These are appended AFTER the ranked candidates, so they can never
        displace a scored one.
        """
        target = self.pool.neutral_mass(mz, adduct, self.offset_ppm)
        if target is None:
            return out, scores
        wide = self.pool.query(mz, adduct, ppm=ppm * 20, offset_ppm=self.offset_ppm)
        if len(wide) == 0:
            return out, scores

        # np.isin(..., invert=True) drops the ones already ranked.
        extra = wide[np.isin(wide, out, invert=True)]
        if len(extra) == 0:
            return out, scores

        extra = extra[np.argsort(np.abs(self.pool.mass[extra] - target))]
        extra = extra[: k - len(out)]
        return (
            np.concatenate([out, extra]),
            np.concatenate([scores, np.zeros(len(extra), dtype=np.float32)]),
        )


In [ ]:
%%writefile casmi/features.py
"""Per-candidate evidence and features for a gradient-boosted ranker (Phase 8).

The public engine beat our pipeline with channels no better than ours: it fed
RAW evidence to a learned combiner, kept a library channel, and trained on
natural-product simulations (docs/13). This module is our version of that
recipe, built only from our own components. It is the single implementation of
the features: the harness that trains and evaluates the ranker and the Kaggle
notebook that serves it both call it, so they cannot drift (docs/07, docs/15).

Two steps, because the spectral searches are slow and the model is not:
    ev = molecule_evidence(lib, pool, spectra, adduct, exclude=...)   # searches
    ev["fp"] = fp_evidence(model, ck, ev)                             # fpnet
    X = candidate_features(ev)                                        # cheap

Ties inside a molecule are AVERAGED when ranking (rank01): breaking them by
list position was a train/serve skew in the public engine (docs/15).
"""

from __future__ import annotations

import numpy as np
from scipy.stats import rankdata

from casmi.analog import rank_by_analogy
from casmi.fpnet import candidate_bits, predict_logits, score_candidates
from casmi.fragment import score_spectra_frag
from casmi.library import Library
from casmi.pool import ADDUCT_MASS, CandidatePool
from casmi.rank import MZ_OFFSET_PPM, PPM, TOP_ANALOGS

# Per-candidate channels; each becomes 4 columns (raw, rank, gap, z).
CHANNELS = ["id_max", "id_mean", "hyb_self", "n_lib", "analog", "fp", "frag"]
CONTEXT = ["ppm", "abs_ppm", "log_n_cand", "an_conf", "id_top", "hyb_top",
           "n_spectra", "log_max_peaks"]
FEATURE_NAMES = ([f"{c}_{t}" for c in CHANNELS for t in ("raw", "rank", "gap", "z")]
                 + CONTEXT)


def rank01(x: np.ndarray) -> np.ndarray:
    """0 = best, 1 = worst within the molecule; tied values share their average."""
    x = np.asarray(x, dtype=np.float64)
    return ((rankdata(-x, method="average") - 1) / max(1, len(x) - 1)).astype(np.float32)


def _z(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    s = x.std()
    return ((x - x.mean()) / s if s > 1e-12 else np.zeros_like(x)).astype(np.float32)


def _row_of_key(lib: Library) -> dict:
    """{InChIKey14: library structure index}, built once per Library."""
    if not hasattr(lib, "_row_of_key"):
        lib._row_of_key = {k: i for i, k in enumerate(lib.structures.inchikey14.to_numpy())}
    return lib._row_of_key


def molecule_evidence(lib: Library, pool: CandidatePool, spectra, adducts,
                      exclude: np.ndarray | None = None, ppm: float = PPM,
                      offset_ppm: float = MZ_OFFSET_PPM, instrument_id: int = 0,
                      collision_energies=None) -> dict:
    """Everything the searches say about one molecule's candidates.

    spectra: [(precursor_mz, peaks (n, 2)), ...], all measurements of the
    molecule; adducts: one per spectrum -- a molecule is often measured as
    several ([M+H]+ and [M-H]-: 160 of the 250 enveda-np-examples structures).
    The candidate window is centred on the MEDIAN neutral mass over spectra.
    exclude: library positions to hide (the class-1/2 simulations).
    Returns a dict of per-candidate arrays plus the query itself (for fpnet).
    """
    masses = [pool.neutral_mass(mz, a, offset_ppm) for (mz, _), a in zip(spectra, adducts)]
    masses = [x for x in masses if x is not None and x > 0]
    m = float(np.median(masses)) if masses else None
    if m is None:
        cand = np.empty(0, np.int64)
    else:
        lo, hi = np.searchsorted(pool.mass, [m * (1 - ppm * 1e-6), m * (1 + ppm * 1e-6)])
        cand = np.arange(lo, hi, dtype=np.int64)
    ev = {"cand": cand, "spectra": spectra, "adducts": list(adducts),
          "instrument_id": instrument_id, "collision_energies": collision_energies}
    if len(cand) == 0:
        return ev

    # Identity (same precursor) and hybrid (any precursor shift) per spectrum,
    # each collapsed to one score per library structure.
    ident = np.zeros((len(spectra), lib.n_structures), np.float32)
    hyb = np.zeros(lib.n_structures, np.float32)
    for j, (q_mz, peaks) in enumerate(spectra):
        ident[j] = lib.score_spectrum(q_mz, peaks, exclude=exclude)
        h = lib.hybrid_scores(q_mz, peaks, exclude)
        if h is not None:
            np.maximum(hyb, h, out=hyb)

    # Library spectra each structure still has after hiding.
    n_lib = np.diff(lib._starts).astype(np.float32)
    if exclude is not None and len(exclude):
        n_lib -= np.bincount(lib.struct_idx[exclude], minlength=lib.n_structures)

    # Map every per-structure array onto the candidates (-1 = not in library).
    rok = _row_of_key(lib)
    row = np.array([rok.get(k, -1) for k in pool.key[cand]])
    has = row >= 0

    def on_cand(per_struct):
        v = np.zeros(len(cand), np.float32)
        v[has] = per_struct[row[has]]
        return v

    ev["id_max"] = on_cand(ident.max(0))
    ev["id_mean"] = on_cand(ident.mean(0))
    ev["hyb_self"] = on_cand(hyb)
    ev["n_lib"] = np.log1p(on_cand(n_lib))

    # Analog: the top cousins by hybrid score, propagated by Tanimoto -- the
    # same channel as rank.py. A candidate's own entry is included here as in
    # rank.py; hyb_self lets the ranker tell "is a cousin" from "is itself".
    n = min(TOP_ANALOGS, lib.n_structures)
    a_idx = np.argpartition(-hyb, n - 1)[:n]
    a_idx = a_idx[np.argsort(-hyb[a_idx])]
    ev["analog"] = rank_by_analogy(pool.smiles[cand], lib.smiles[a_idx], hyb[a_idx])
    ev["an_conf"] = float(hyb[a_idx[0]])

    # score_spectra_frag uses the richest spectrum; give it that one's adduct.
    j = max(range(len(spectra)), key=lambda i: len(spectra[i][1]))
    ev["frag"] = score_spectra_frag(pool.smiles[cand], [spectra[j]], adducts[j]).astype(np.float32)
    ev["ppm"] = ((pool.mass[cand] - m) / m * 1e6).astype(np.float32)
    return ev


def fp_evidence(model, ck, ev: dict, pool: CandidatePool, device: str = "cpu") -> np.ndarray:
    """fpnet log-likelihood of each candidate (Part 5), for one molecule.

    predict_logits takes one adduct, so spectra are grouped by adduct and the
    group logits averaged, weighted by group size -- the same per-spectrum
    mean predict_logits takes within a group.
    """
    if len(ev["cand"]) == 0:
        return np.zeros(0, np.float32)
    ces = ev["collision_energies"] or [None] * len(ev["spectra"])
    logits, n = 0.0, 0
    for a in sorted(set(ev["adducts"])):
        idx = [i for i, x in enumerate(ev["adducts"]) if x == a]
        sub_ces = [ces[i] for i in idx]
        z = predict_logits(model, [ev["spectra"][i] for i in idx], a, list(ck["adduct_list"]),
                           instrument_id_=ev["instrument_id"], device=device,
                           max_peaks=int(ck["max_peaks"]),
                           collision_energies=None if None in sub_ces else sub_ces)
        logits, n = logits + z * len(idx), n + len(idx)
    bits = candidate_bits(pool.smiles[ev["cand"]], np.asarray(ck["bit_sel"]))
    return score_candidates(logits / n, bits).astype(np.float32)


def candidate_features(ev: dict) -> np.ndarray:
    """[n_candidates, len(FEATURE_NAMES)] float32, in FEATURE_NAMES order.

    Raw values keep how STRONG the evidence is (a fixed z-score blend threw
    that away); rank/gap/z say how a candidate compares with its competitors.
    A channel that was not computed (e.g. no fpnet) is all zeros.
    """
    nc = len(ev["cand"])
    cols = []
    for c in CHANNELS:
        v = np.asarray(ev.get(c, np.zeros(nc)), np.float32)
        cols += [v, rank01(v), v - (v.max() if nc else 0.0), _z(v)]
    ppm = ev["ppm"]
    ctx = [ppm, np.abs(ppm),
           np.full(nc, np.log(max(nc, 1))),
           np.full(nc, ev["an_conf"]),
           np.full(nc, ev["id_max"].max()),
           np.full(nc, ev["hyb_self"].max()),
           np.full(nc, len(ev["spectra"])),
           np.full(nc, np.log1p(max(len(p) for _, p in ev["spectra"])))]
    return np.column_stack(cols + ctx).astype(np.float32)


# --- the ranker ---------------------------------------------------------------
# Gradient-boosted trees, the public engine's settings. One classifier per
# (class-1 weight, seed), averaged: class-1 and class-2 simulation rows pull in
# opposite directions on "has library spectra" (docs/16), and the weight sets
# the balance. Trained in the notebook from shipped rows, so no pickled model
# has to survive a scikit-learn version change.
GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80,
           l2_regularization=1.0)
W_A = (0.35, 0.55)
SEEDS = (0, 1)


def fit_ranker(X: np.ndarray, y: np.ndarray, is_c1: np.ndarray, monotonic_cst=None,
               w_a=W_A) -> list:
    """Train the ranker on simulation rows (y: is the true candidate).

    w_a: weights of the class-1 rows (class-2 rows get 1 - w), one model each.
    """
    from sklearn.ensemble import HistGradientBoostingClassifier
    return [HistGradientBoostingClassifier(random_state=sd, monotonic_cst=monotonic_cst, **GBM)
            .fit(X, y, sample_weight=np.where(is_c1, wA, 1.0 - wA))
            for wA in w_a for sd in SEEDS]


def ranker_scores(models: list, X: np.ndarray) -> np.ndarray:
    """Mean probability of being the true candidate; higher ranks first."""
    return np.mean([m.predict_proba(X)[:, 1] for m in models], axis=0)


def _self_check() -> None:
    # Ties share their rank: two zeros among three values both get 0.75.
    r = rank01(np.array([1.0, 0.0, 0.0]))
    assert np.allclose(r, [0.0, 0.75, 0.75]), r
    assert len(FEATURE_NAMES) == 4 * len(CHANNELS) + len(CONTEXT)
    ev = {"cand": np.arange(3), "spectra": [(200.0, np.zeros((5, 2)))],
          "ppm": np.array([0.1, -1.0, 2.0], np.float32), "an_conf": 0.4,
          "id_max": np.array([0.9, 0.0, 0.0], np.float32),
          "hyb_self": np.array([0.9, 0.2, 0.0], np.float32)}
    X = candidate_features(ev)
    assert X.shape == (3, len(FEATURE_NAMES)) and np.isfinite(X).all()
    print("features self-check OK")


if __name__ == "__main__":
    _self_check()


In [ ]:
import gc, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from casmi.library import Library, build, TEST_MZ_RANGE, TEST_ADDUCTS
from casmi.pool import CandidatePool
from casmi.fpnet import load
from casmi.features import (FEATURE_NAMES, candidate_features, fit_ranker, fp_evidence,
                            molecule_evidence, rank01, ranker_scores)

t_own = time.time()
OWN_INDEX = Path("/tmp/own_index")
build(Path(COMP) / "train.parquet", OWN_INDEX, TEST_MZ_RANGE, set(TEST_ADDUCTS))
lib, pool = Library(OWN_INDEX), CandidatePool(Path(find("pool.parquet")))
dev = "cuda" if torch.cuda.is_available() else "cpu"
fpnet, fck = load(find("fpnet.pt"), device=dev)
assert fck.get("model_tag") == "v5", f"stale casmi26-deps: fpnet {fck.get('model_tag')}"
R = np.load(find("ranker_rows.npz"), allow_pickle=True)
assert str(R["fpnet_tag"]) == "v5" and list(R["feature_names"]) == FEATURE_NAMES
own_models = fit_ranker(R["X"], R["y"], R["c1"])

OWN = {}   # molecule_id -> (candidate InChIKey14s, our ranker's scores)
te_own = pd.read_parquet(Path(COMP) / "test.parquet")
for mid, grp in te_own.groupby("molecule_id", sort=True):
    spectra = [(float(r.precursor_mz),
                np.stack([np.asarray(r.ms2_mzs, np.float32),
                          np.asarray(r.ms2_normalized_intensities, np.float32)], 1))
               for r in grp.itertuples()]
    ces = [float(np.mean(r.collision_energy_ev))
           if r.collision_energy_ev is not None and len(r.collision_energy_ev) else 32.0
           for r in grp.itertuples()]
    ev = molecule_evidence(lib, pool, spectra, list(grp.adduct), instrument_id=0,
                           collision_energies=ces)
    if len(ev["cand"]):
        ev["fp"] = fp_evidence(fpnet, fck, ev, pool, device=dev)
        OWN[mid] = (pool.key[ev["cand"]].astype(str),
                    ranker_scores(own_models, candidate_features(ev)))
print(f"our pipeline: {len(OWN)}/{te_own.molecule_id.nunique()} molecules scored "
      f"in {(time.time() - t_own) / 60:.1f} min")
del lib, pool, fpnet, own_models, R, te_own
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import numpy as np, pandas as pd
import casmi_engine as E
E.RANK.W_A = (0.35, 0.55)
E.RANK.SEEDS = (0, 1)
fp_models = sorted(glob.glob('/kaggle/input/**/fp_*.pt', recursive=True))
print('fp models:', fp_models)
# ---- blend our ranker into the engine's final ranking (ryqn00) ----------------
W_OWN = 0.1
_make_submissions = E.make_submissions
BLEND_STATS = {"blended": 0, "engine only": 0}

def make_submissions_with_own(mols, recs, score_sets, *a, **k):
    """final = (1 - W_OWN) * engine rank + W_OWN * our rank, within each molecule.
    Engine candidates outside our window get our worst rank (-1 < any probability)."""
    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}
    out = {}
    for mid, s in score_sets["blend"].items():
        if mid not in OWN:
            out[mid] = s
            BLEND_STATS["engine only"] += 1
            continue
        ok = dict(zip(*OWN[mid]))
        so = np.array([ok.get(key, -1.0) for key in E.POOL["keys"][cand[mid]].astype(str)])
        out[mid] = (1 - W_OWN) * (1 - rank01(s)) + W_OWN * (1 - rank01(so))
        BLEND_STATS["blended"] += 1
    return _make_submissions(mols, recs, dict(score_sets, blend=out), *a, **k)

E.make_submissions = make_submissions_with_own
subs, recs = E.main(os.path.join(COMP, 'test.parquet'), os.path.join(COMP, 'train.parquet'),
                    os.path.join(COMP, 'sample_submission.csv'), find('sim_rank_rows_nofp.npz'), find('rank_train.npz'),
                    fp_models, workers=os.cpu_count(), w_pv=0.88)
print('blend:', BLEND_STATS)
subs['blend'].to_csv('submission.csv', index=False)
subs['pv'].to_csv('submission_pv.csv', index=False)
subs['ours'].to_csv('submission_ours.csv', index=False)
print({k: v.shape for k, v in subs.items()}, 'total %.0f min' % ((time.time() - T_START) / 60))
subs['blend'].head()

## 5 · Numbers behind the choices

Predicted LB uses prvsiyan's calibration `0.162·c1 + 0.220·c2` on enveda-np-examples (the only held-out timsTOF set).

| Features (no fingerprint channel, trained on 7 sets) | np Class-1 | np Class-2 | predicted LB |
|---|---|---|---|
| prvsiyan's 31 features | 0.852 | 0.644 | 0.280 |
| + adduct-shifted / mean / count library block | **0.872** | **0.655** | **0.285** |
| + polarity analogs | 0.856 | 0.651 | 0.282 |
| all blocks | 0.871 | 0.646 | 0.283 |

Channel-only MRR, Class-2 simulation (enveda-np-examples): analog 0.524 · same-polarity analog 0.532 · public fingerprint model 0.493 · MetFrag-lite 0.243. Class-1: library 0.930 · adduct-shifted library 0.938.

**Dead end measured along the way:** DreaMS embeddings as an analog channel tie the classical mass-shifted search on Class-2 (0.58 vs 0.57–0.63) and only help Class-1.

In [ ]:
print('EXPERIMENT_PUBLIC_RANKER_WEIGHT=0.88; rank features: ties averaged; + 0.1 x ryqn00 ranker')
print('submission rows:', len(subs['blend']))